# 🧬 00 · Welcome: what machine learning does for biology
### From one gene at a time to genome-scale tables — a guided tour of 02-620 with leukaemia-like expression data, where you only have to press Shift + Enter

**BiologyToML** · Level 1 · 02-620 core · Part A — Foundations: from biology to data · Module 00 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/00_welcome_ml_for_biology.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, pandas, Matplotlib (no installation needed on Colab) |
| **Prerequisites** | None. High-school algebra and introductory biology. You will only *run* code here; Module 01 teaches you to write it. |
| **Estimated time** | 85 min (reading + running + the ✏️ exercises) |
| **02-620 topics covered** | Introduction — machine learning in science: where machine learning comes from (statistics meets computer science, learning from examples); how genome sequencing, maps of human variation and expression profiling made biology a data science; supervised learning (f: X → Y, regression vs classification, modelling P(Y | X)); unsupervised learning (modelling P(X)); learning a joint distribution; MLE vs MAP and Bayes' rule (preview); the plan of the course |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> **All data are SIMULATED.** The leukaemia expression table is generated by the toolkit's
>     `simulate_leukemia_expression`: 72 patients × 3,571 genes shaped like the classic study of Golub *et al.* (1999) —
>     same class sizes, training/test split, measurement scale and marker-gene names — but the values are simulated and do
>     not reproduce the paper's numbers. The DNA sequence and SNPs of Section 2 and the population map of Section 5 are
>     simulated too, each with a known truth. The tiny two-gene classifier of Section 4 is a teaching toy, not a
>     diagnostic tool.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Run** a Jupyter/Colab notebook confidently: execute cells in order, restart, and recover from mistakes.
2. **Describe** the three breakthroughs that made biology a data science (genome sequence, maps of human variation, expression profiling) and the machine-learning question each one raised.
3. **Explain** why biological data are stored as a table with **N rows = samples** and **J columns = features**, and read the shape of a data set.
4. **Distinguish** supervised learning (learn f: X → Y, or P(Y|X)) from unsupervised learning (learn P(X)), and regression from classification.
5. **Interpret** a predicted probability such as P(AML | expression) = 0.9 vs 0.5 as a statement of certainty.
6. **Identify** X, Y and f for gene finding, GWAS, cancer subtyping, gene networks and population structure.
7. **State** how maximum-likelihood (MLE) and maximum a posteriori (MAP) estimation differ, and how Bayes' rule combines a prior with the likelihood.
8. **Connect** every lecture topic of 02-620 to the module of this course that teaches it, and name the three ideas that recur in all of them.
9. **Navigate** the two levels of the course — the 02-620 core (Modules 00–23) and the research frontier (Modules 24–43) — and the capstone (44).

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | check the computer, load the helpers | 5 min |
| 1 · How to use this course | notebooks, cells, the emoji legend, the 45-module roadmap, 02-620 at a glance | 15 min |
| 2 · Biology became a data science | where machine learning came from; three breakthroughs; a simulated gene and SNPs; how big the data are | 15 min |
| 3 · Data as tables | leukaemia-like data, 72 patients × 3,571 genes; the classic ALL/AML heatmap; 🧪 true marker genes | 10 min |
| 4 · Supervised learning | the learning workflow; a two-gene classifier that outputs probabilities | 12 min |
| 5 · Unsupervised learning | structure without labels: maps of patients and of a simulated population; 🧪 true subtypes and places | 10 min |
| 6 · What is X? What is Y? What is f? | six biological problems in one framework | 6 min |
| 7 · The course map | methods × tasks, joint distributions, MLE vs MAP, the three recurring ideas, the 02-620 schedule | 10 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 5 min |

## 🧫 The biological question

In 1999 a patient arrives at a hospital with acute leukaemia: the bone marrow is flooded with immature white blood
cells. Two diseases look alike at this stage. **Acute lymphoblastic leukaemia (ALL)** grows from precursors of
lymphocytes; **acute myeloid leukaemia (AML)** grows from precursors of the myeloid line (granulocytes, monocytes).
They need **different treatments**, so telling them apart matters. Yet no single test settled the question: the
diagnosis combined an experienced haematopathologist's reading of cell shape, staining, surface markers and
chromosomes, each done in a different specialised laboratory.

Todd Golub, Donna Slonim, Eric Lander and colleagues asked a new kind of question. A **microarray** can measure the
activity (expression) of *thousands* of genes in one sample at once. Could a computer **learn** the difference between
ALL and AML from such measurements on patients whose diagnosis was known, and then **predict** the class of new
patients? And could it **discover** the two classes on its own, without being told they exist? Their paper,
*"Molecular classification of cancer: class discovery and class prediction by gene expression monitoring"* (Science,
1999), became one of the founding papers of machine learning in biology, and it is the first case study of 02-620.

This notebook is your guided tour. You will explore a **simulated data set shaped like Golub's** — the same number of
patients and genes, the same marker-gene names — see the pattern the authors saw, build a tiny classifier that outputs
probabilities, and watch the two leukaemias separate without labels. Along the way you will learn how this course
works, how its 24 core modules map onto the lecture topics of 02-620, and how 20 frontier modules then carry the same
ideas to current computational-biology research.

> ▶️ **Never used a notebook before?** A notebook is a document made of *cells*. Grey boxes are **code cells**: click
> inside one and press **Shift + Enter** (or the ▶ button on its left) to run it; its output appears underneath. White
> boxes like this one are **text cells**. Run the cells **from top to bottom** — each one may use results of the cells
> above it. On Google Colab, click the *Open in Colab* badge at the top, then *Runtime → Run all* if you just want to see
> everything once. Nothing you do here can break your computer, and every mistake can be undone by re-running.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("pandas", "pandas"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
`simulate_leukemia_expression` generates the course's leukaemia-like expression data (with a
               fixed seed, so everyone gets the same numbers, and together with the hidden truth);
               `standardize` puts genes on a common scale; `accuracy` and `confusion_counts` score
               predictions; `plot_decision_regions` draws what a classifier decides. You do not need to read
               this code today. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import numpy as np
import matplotlib.pyplot as plt

LEUKEMIA_MARKERS = {
    "CST3": ("AML", "cystatin C"), "ZYX": ("AML", "zyxin"), "CD33": ("AML", "CD33 antigen"),
    "MPO": ("AML", "myeloperoxidase"), "LYZ": ("AML", "lysozyme"), "CTSD": ("AML", "cathepsin D"),
    "FAH": ("AML", "fumarylacetoacetate hydrolase"), "LTC4S": ("AML", "leukotriene C4 synthase"),
    "LYN": ("AML", "LYN kinase"), "HOXA9": ("AML", "homeobox A9"), "CFD": ("AML", "adipsin"),
    "LEPR": ("AML", "leptin receptor"), "SRGN": ("AML", "proteoglycan 1 (serglycin)"),
    "CXCL8": ("AML", "interleukin-8"), "AZU1": ("AML", "azurocidin"), "MCL1": ("AML", "MCL1"),
    "ITGAX": ("AML", "CD11c"), "NFKBIA": ("AML", "MAD-3 (IκBα)"), "ELANE": ("AML", "neutrophil elastase"),
    "CEBPA": ("AML", "C/EBPα"), "CSF3R": ("AML", "G-CSF receptor"), "FCER1G": ("AML", "Fc receptor γ"),
    "TYROBP": ("AML", "DAP12"), "S100A9": ("AML", "calprotectin subunit"), "CFP": ("AML", "properdin"),
    "MYB": ("ALL", "c-Myb"), "CCND3": ("ALL", "cyclin D3"), "RBBP4": ("ALL", "RbAp48"),
    "SMARCA4": ("ALL", "SNF2 (BRG1)"), "TCF3": ("ALL", "E2A"), "DUT": ("ALL", "dUTPase"),
    "STMN1": ("ALL", "Op18 (stathmin)"), "IL7R": ("ALL", "IL-7 receptor"), "RABEP1": ("ALL", "rabaptin-5"),
    "CBX3": ("ALL", "heterochromatin protein p25"), "DNTT": ("ALL", "terminal deoxynucleotidyl transferase"),
    "CD24": ("ALL", "CD24"), "TOP2B": ("ALL", "topoisomerase II β"), "IRF2": ("ALL", "IRF-2"),
    "MYL6": ("ALL", "myosin light chain"), "PSMB8": ("ALL", "proteasome subunit"), "LMO2": ("ALL", "LMO2"),
    "CD79A": ("B-ALL", "MB-1 (Igα)"), "CD19": ("B-ALL", "CD19"), "CD79B": ("B-ALL", "Igβ"),
    "VPREB1": ("B-ALL", "pre-B receptor"), "PAX5": ("B-ALL", "PAX5"), "CD22": ("B-ALL", "CD22"),
    "CD3E": ("T-ALL", "CD3ε"), "CD3D": ("T-ALL", "CD3δ"), "CD2": ("T-ALL", "CD2"), "CD7": ("T-ALL", "CD7"),
    "LCK": ("T-ALL", "LCK kinase"), "CD1E": ("T-ALL", "CD1e"),
}


def simulate_leukemia_expression(seed: int = 1999) -> dict:
    """Synthetic leukaemia expression data shaped like the classic Golub et al. (1999) study.

    72 bone-marrow/blood samples — 47 acute lymphoblastic leukaemia (ALL: 38 B-lineage, 9 T-lineage)
    and 25 acute myeloid leukaemia (AML) — × 3,571 genes on a log10 microarray scale, with the
    original 38/34 training/test design (train: 19 B-ALL, 8 T-ALL, 11 AML).  Values below the
    detection limit are floored at 2.0 (≈ 23 % of entries) and saturate at 4.2.  The 54 genes of
    ``LEUKEMIA_MARKERS`` carry the strongest signals (42 differ between ALL and AML by 0.5–1.0
    log10 units; 12 mark B- or T-lineage ALL), 900 more genes differ modestly between ALL and AML,
    150 separate T-ALL from B-ALL, and 15 hidden co-expression programmes correlate genes.  Two
    AML samples are atypical (signature at 35 % strength).  The classes are cleaner than in the
    real study, so many classifiers make few or no errors.

    Returns a dict with ``X`` (72 × 3,571), ``y`` (0 = ALL, 1 = AML), ``labels``, ``subtype``,
    ``genes``, ``is_train``, ``sample_id`` (1–72), ``atypical`` (True for the two weak-signature AML
    samples), ``true_effect`` and ``source``.  ``true_effect`` is the planted AML − ALL difference
    per gene; it is 0 for the 12 lineage markers and the 150 T-vs-B genes, although the B-lineage
    markers still differ between ALL and AML through the subtype shifts (by about 0.8 log10 units).
    The default ``seed`` gives the course's standard data set.
    """
    rng = np.random.default_rng(seed)
    J = 3571
    subtype = np.array(["B-ALL"] * 19 + ["T-ALL"] * 8 + ["AML"] * 11 + ["B-ALL"] * 19 + ["T-ALL"] * 1 + ["AML"] * 14,
                       dtype=object)
    is_train = np.r_[np.ones(38, bool), np.zeros(34, bool)]
    order = np.r_[rng.permutation(38), 38 + rng.permutation(34)]          # mix classes within each set
    subtype, is_train = subtype[order], is_train[order]
    n = len(subtype)
    y = (subtype == "AML").astype(int)
    named = list(LEUKEMIA_MARKERS)
    genes = np.array(named + [f"G{j:04d}" for j in range(1, J - len(named) + 1)], dtype=object)
    low = rng.random(J) < 0.4                                             # genes mostly below detection
    mu = np.where(low, rng.uniform(1.5, 2.4, J), np.clip(rng.normal(2.9, 0.4, J), 2.2, 4.0))
    sd = rng.uniform(0.12, 0.35, J)
    X = mu + sd * rng.normal(0, 1, (n, J))
    factors = rng.normal(0, 1, (n, 15))                                  # co-expression programmes
    loadings = rng.normal(0, 0.15, (15, J)) * (rng.random((15, J)) < 0.08)
    X += factors @ loadings
    effect = np.zeros(J)
    strong = np.arange(len(named))
    for j, g in enumerate(named):
        cls = LEUKEMIA_MARKERS[g][0]
        size = rng.uniform(0.5, 1.0)
        effect[j] = size if cls == "AML" else -size if cls == "ALL" else 0.0
        mu[j] = max(mu[j], 2.5)
    moderate = rng.choice(np.arange(len(named), J), 900, replace=False)
    effect[moderate] = rng.choice([-1, 1], 900) * rng.uniform(0.1, 0.4, 900)
    strength = np.ones(n)
    strength[np.flatnonzero(y == 1)[rng.choice((y == 1).sum(), 2, replace=False)]] = 0.35   # atypical AML
    X[:, strong] = mu[strong] + sd[strong] * rng.normal(0, 1, (n, len(strong)))
    X += np.outer((y - y.mean()) * strength, effect)
    for g, (cls, _) in LEUKEMIA_MARKERS.items():
        if cls in ("B-ALL", "T-ALL"):
            j = named.index(g)
            X[:, j] += np.where(subtype == cls, 0.9, np.where(subtype == "AML", -0.2, -0.5))
    lineage = rng.choice(np.setdiff1d(np.arange(len(named), J), moderate), 150, replace=False)
    X[:, lineage] += np.outer(np.where(subtype == "T-ALL", 1.0, np.where(subtype == "B-ALL", -0.2, 0.0)),
                              rng.choice([-1, 1], 150) * rng.uniform(0.4, 0.8, 150))
    X = np.round(np.clip(X, 2.0, np.log10(16000)), 4)
    return dict(X=X, y=y, labels=np.where(y == 1, "AML", "ALL").astype(object), subtype=subtype, genes=genes,
                is_train=is_train, sample_id=np.arange(1, n + 1), atypical=strength < 1, true_effect=effect,
                source="SIMULATED leukaemia data (shaped like Golub et al. 1999)")


def standardize(X, mean=None, sd=None):
    """Z-score the columns of X. Pass the training mean/sd to transform test data the same way.

    Returns ``(Z, mean, sd)``; constant columns get sd = 1 so they map to 0.
    """
    X = np.asarray(X, float)
    mean = X.mean(axis=0) if mean is None else mean
    sd = X.std(axis=0) if sd is None else sd
    sd = np.where(sd > 0, sd, 1.0)
    return (X - mean) / sd, mean, sd


def accuracy(y_true, y_pred) -> float:
    """Fraction of correct predictions."""
    return float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))


def confusion_counts(y_true, y_pred) -> dict:
    """True/false positives/negatives for binary labels (1 = positive)."""
    y_true, y_pred = np.asarray(y_true).astype(int), np.asarray(y_pred).astype(int)
    return dict(TP=int(np.sum((y_pred == 1) & (y_true == 1))), FP=int(np.sum((y_pred == 1) & (y_true == 0))),
                FN=int(np.sum((y_pred == 0) & (y_true == 1))), TN=int(np.sum((y_pred == 0) & (y_true == 0))))


CLASS_COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e", "#8c564b"]


def plot_decision_regions(predict, X, y, ax=None, resolution: int = 200, title: str = "",
                          class_names=None, pad: float = 0.5):
    """Shade the regions a 2-feature classifier assigns to each class and overlay the data.

    ``predict`` maps an (M, 2) array to M integer labels 0, 1, 2, …
    """
    X, y = np.asarray(X, float), np.asarray(y).astype(int)
    ax = ax or plt.gca()
    x0, x1 = X[:, 0].min() - pad, X[:, 0].max() + pad
    y0, y1 = X[:, 1].min() - pad, X[:, 1].max() + pad
    gx, gy = np.meshgrid(np.linspace(x0, x1, resolution), np.linspace(y0, y1, resolution))
    Z = np.asarray(predict(np.c_[gx.ravel(), gy.ravel()])).astype(int).reshape(gx.shape)
    k = max(int(y.max()), int(Z.max())) + 1
    from matplotlib.colors import ListedColormap
    cmap = ListedColormap(CLASS_COLORS[:k])
    ax.contourf(gx, gy, Z, levels=np.arange(k + 1) - 0.5, cmap=cmap, alpha=0.18)
    ax.contour(gx, gy, Z, levels=np.arange(k) + 0.5, colors="k", linewidths=0.8)
    for c in range(k):
        m = y == c
        if m.any():
            ax.scatter(X[m, 0], X[m, 1], s=18, color=CLASS_COLORS[c], edgecolor="k", linewidth=0.3,
                       label=(class_names[c] if class_names is not None else f"class {c}"))
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
    if title:
        ax.set_title(title)
    return ax


print("🧰 Toolkit loaded:", ", ".join(['simulate_leukemia_expression', 'standardize', 'accuracy', 'confusion_counts', 'plot_decision_regions']))

---
## 1 · How to use this course

> 📍 **Where we are.** This is the very first module. Before any biology or mathematics, let us make sure the tool —
> the notebook — feels comfortable, and see where the 45 modules are going.

### 1.1 Cells, the kernel and the order of execution

Behind every notebook runs a Python program called the **kernel**. When you run a code cell, the kernel executes it
and *remembers* everything the cell created (numbers, tables, functions). The number in square brackets to the left
of a cell, e.g. `[3]`, tells you the order in which cells were run. Three habits will save you hours:

1. **Run cells in order, from the top.** A cell that uses a variable defined further up will fail if the cell above
   has not been run yet.
2. **If things get strange, restart.** *Runtime → Restart and run all* (Colab) or *Kernel → Restart & Run All*
   (Jupyter) wipes the kernel's memory and runs everything again from a clean state.
3. **Errors are normal.** A red error message is the computer telling you exactly what went wrong; Module 01 teaches
   you to read them. Fix, re-run, move on.

Try it: run the next cell. It does a little genome arithmetic — the human genome has about 3 billion bases (the size
of the first draft sequence of 2001, Section 2) and roughly 20,000 protein-coding genes.

In [ ]:
# ── 1.1 Your first code cell: genome arithmetic ──
genome_bases = 3_000_000_000          # ≈ 3 billion bases in one copy of the human genome
protein_coding_genes = 20_000         # roughly 20,000 protein-coding genes
print("Hello, biologist!")
print("Bases of genome per protein-coding gene, on average:", genome_bases // protein_coding_genes)

🔍 **What just happened?** The kernel stored two numbers under the names `genome_bases` and `protein_coding_genes`
(such names are called **variables**), divided one by the other and printed the result: about **150,000 bases per
gene**. Genes are scattered sparsely through a sea of non-coding DNA — which is exactly why *finding* them (Section 2)
was a hard problem. Lines starting with `#` are **comments**: notes for humans that the computer ignores.

In [ ]:
# ── 1.2 Cells remember: use the variables defined above ──
n_ALL, n_AML = 47, 25                 # patients of each leukaemia type in the Golub study (and in our simulation)
n_patients = n_ALL + n_AML
print(f"{n_patients} patients, of whom {n_AML / n_patients:.0%} have AML")
print(f"Bases in one copy of the genome: {genome_bases:,}  (defined in the cell above — the kernel remembered it)")

🔍 **What just happened?** This cell used `genome_bases`, which was created by the *previous* cell — the kernel
remembers it. The odd-looking `f"…{…}"` is a formatted string: Python replaces each `{…}` by its value (`:.0%` shows a
fraction as a percentage, `:,` adds thousands separators).

⚠️ **Order matters.** If you restart the kernel and run cell 1.2 *before* cell 1.1, Python stops with
`NameError: name 'genome_bases' is not defined`. Not a disaster: run cell 1.1, then 1.2 again.

✏️ **Your turn.** In cell 1.2 change `n_AML` to `30` and re-run it with Shift + Enter. The percentage changes. Now set
it back to `25` and re-run, so that later cells see the original number. (You have just done what scientists do all
day: change one thing, re-run, compare.)

### 1.2 The anatomy of every notebook and the emoji legend

All 45 notebooks share one structure, so you always know where to look:

| Symbol | Meaning |
|---|---|
| 🎯 / 🗺️ | learning objectives and the contents table (with minutes per section) |
| 🧫 | the biological question that motivates the module |
| 📍 **Where we are** | how a section connects to what came before |
| 🧮 **Math refresher** | a gentle (re)introduction to a piece of mathematics, assuming you may never have seen it |
| *In words:* | a plain-language reading that follows **every** equation |
| 🔍 **What just happened?** | interpretation of the output you just produced — compare it with what *you* see |
| ✏️ **Your turn** | a concrete change for you to make, and a question to answer |
| ⚠️ | a pitfall or common misconception |
| 🧪 **Ground-truth check** | simulated data come with a known truth, so we check whether a method actually found it |
| 📘 **Lecture notes** | a self-contained summary of the part of the 02-620 lecture a section teaches — definitions, key equations, a worked example and an exam tip (core modules) |
| 📄 **Research connection** | what the research field does, what the notebook builds, and where the frontier is (frontier modules) |
| ✅ / 🧠 / 🚀 / 📝 | checkpoint questions with hidden answers, open questions, extensions, and a lab-note template |

**A study workflow that works.** (1) Run the whole notebook once, top to bottom, reading the text. (2) Go back section
by section; re-run each cell and check that the 🔍 text matches your output. (3) Do every ✏️ — *change one parameter
at a time* and predict the result before you run. (4) Fill in the 📝 lab note at the end: writing down what changed
when you modified something is how the material sticks. (5) Before an exam, revise from the 📘 Lecture notes boxes:
together they summarise everything the 02-620 lecture on the topic teaches.

### 1.3 The roadmap

The course has five parts. **Part A** (Modules 00–05) teaches the prerequisites of 02-620 — Python, probability,
linear algebra, calculus — *just in time*, always with biological data. Parts B–E follow the 02-620 lectures. Run the
next cell to draw the map.

In [ ]:
# ── 1.3 Draw the 45-module roadmap ──
PARTS_MAP = [
    ("A · Foundations", "tab:blue", ["00 Welcome", "01 Python &\nNumPy", "02 Probability", "03 Linear\nalgebra",
                                     "04 Calculus &\noptimization", "05 Likelihood,\nMLE & MAP"]),
    ("B · Supervised", "tab:orange", ["06 Linear\nregression, GWAS", "07 Ridge &\nlasso", "08 k-NN &\nnaïve Bayes",
                                      "09 Logistic\nregression", "10 Trees &\nforests", "11 Support\nvector machines"]),
    ("C · Unsupervised", "tab:green", ["12 Clustering", "13 Mixtures\n& EM", "14 PCA"]),
    ("D · Graphical models", "tab:purple", ["15 Bayes nets I:\nrepresentation", "16 Bayes nets II:\nindependence",
                                            "17 Bayes nets III:\ninference", "18 Gaussian\ngraphical models",
                                            "19 HMM I:\ndecoding", "20 HMM II:\nlearning"]),
    ("E · Bayes, NN, selection", "tab:red", ["21 Bayesian\nlearning, MCMC", "22 Neural\nnetworks",
                                             "23 Model\nselection"]),
    ("F · Modern ML blocks", "tab:brown", ["24 GLMs &\ncount data", "25 Variational\ninference, VAEs",
                                           "26 Attention &\ntransformers", "27 Diffusion &\nflow matching",
                                           "28 Graph neural\nnetworks", "29 GPs & Bayesian\noptimization"]),
    ("G · Cells & tissues", "tab:pink", ["30 Single-cell\nRNA-seq", "31 Batch effects\n& integration",
                                         "32 Cell dynamics\n& velocity", "33 Spatial\nomics",
                                         "34 Multimodal\nintegration"]),
    ("H · Genomes, proteins,\nmolecules", "tab:olive", ["35 Statistical\ngenetics", "36 Regulatory\ngenomics",
                                                         "37 Protein seqs\n& language models",
                                                         "38 Protein\nstructure", "39 Generative\ndesign",
                                                         "40 Molecular ML,\ndrug discovery"]),
    ("I · Causality & trust", "tab:cyan", ["41 Perturbations\n& causality", "42 Trustworthy\nML",
                                           "43 LLMs & AI\nagents"]),
    ("J · Capstone", "tab:gray", ["44 Capstone\nproject"]),
]


def draw_roadmap(parts) -> None:
    """Draw one row of rounded boxes per course part, with arrows from row to row."""
    fig, ax = plt.subplots(figsize=(11.5, 8.6))
    for row, (part, colour, modules) in enumerate(parts):
        y = len(parts) - 1 - row
        ax.text(-0.62, y, part, ha="right", va="center", fontsize=9, fontweight="bold", color=colour)
        for col, name in enumerate(modules):
            ax.add_patch(FancyBboxPatch((col - 0.45, y - 0.36), 0.9, 0.72, boxstyle="round,pad=0.02",
                                        fc=colour, ec="0.3", alpha=0.18 if col else 0.35))
            ax.text(col, y, name, ha="center", va="center", fontsize=7.2)
        if row < len(parts) - 1:            # arrow down to the next part
            ax.annotate("", xy=(0, y - 0.62), xytext=(0, y - 0.38),
                        arrowprops=dict(arrowstyle="-|>", color="0.4"))
    ax.axhline(len(parts) - 5.5, color="0.5", lw=0.8, ls="--")   # Level 1 (rows A–E) above, Level 2 below
    ax.text(-3.15, len(parts) - 5.42, "▲ Level 1 · 02-620 core", ha="left", va="bottom", fontsize=8,
            style="italic", color="0.35")
    ax.text(-3.15, len(parts) - 5.58, "▼ Level 2 · research frontier", ha="left", va="top", fontsize=8,
            style="italic", color="0.35")
    ax.set_xlim(-3.2, 5.6); ax.set_ylim(-0.6, len(parts) - 0.4); ax.axis("off")
    ax.set_title("BiologyToML: 45 modules in ten parts (each row builds on the rows above)")
    plt.tight_layout(); plt.show()


draw_roadmap(PARTS_MAP)
print("modules per part:", {p: len(m) for p, _, m in PARTS_MAP}, "→ total", sum(len(m) for _, _, m in PARTS_MAP))

🔍 **What just happened?** **Level 1 — the 02-620 core** (rows A–E, Modules 00–23): six foundation modules, six
supervised-learning modules, three unsupervised, six on probabilistic graphical models and three on Bayesian learning,
neural networks and model selection. **Level 2 — the research frontier** (rows F–I, Modules 24–43) reuses exactly
these ideas to reach current research: six modules of modern building blocks (count models, variational inference,
transformers, diffusion, graph networks, Gaussian processes), then three application areas — cells and tissues,
genomes/proteins/molecules, and causality/reliability/AI agents. The **capstone** (Module 44) ties everything together:
**45 in total**. You do not need to understand the code of the cell; it only draws boxes.

> 🧭 **Which modules should you do?** Following the 02-620 lectures: Modules 00–23, then the capstone. Aiming at research:
> finish Level 1 first (every frontier module names the core modules it *builds on*), then pick a track — single-cell
> (26, 27, 30–34, 41), genomics (24, 26, 35, 36, 42), proteins and drugs (24–25, 28–29, 37–40) or AI for science (24–25, 27–29, 42,
> 43). The [README](../README.md) and [`FRONTIER_MAP.md`](../FRONTIER_MAP.md) list the tracks; the frontier modules
> use *simulated* data with a known ground truth, so you can always check whether a method found the truth.

**How the modules map onto the 15 lecture topics of 02-620.** Each lecture topic of 02-620 is taught by one or more
modules:

| 02-620 lecture topic | What the lecture covers | Module(s) |
|---|---|---|
| Introduction: machine learning in science | biology as a data science; supervised vs unsupervised learning; P(Y∣X) vs P(X); MLE vs MAP | **00**, 05 (and 44: the course project) |
| Regression | genome-wide association, linear regression, MLE = least squares, ridge (MAP), lasso | 06, 07 (previews in 03, 04) |
| Classification I | k-nearest neighbours, naïve Bayes | 08 |
| Classification II | logistic regression; generative vs discriminative classifiers | 09 |
| Decision trees | decision trees, random forests | 10 |
| Clustering | hierarchical clustering, k-means | 12 |
| Support vector machines | maximum margin, kernels | 11 |
| Probabilistic graphical models | Bayesian networks: representation, conditional independence, inference, learning | 15, 16, 17 |
| Gaussian graphical models | partial correlations, the precision matrix, the graphical lasso | 18 |
| Mixture models | Gaussian mixtures and the EM algorithm | 13 |
| Dimensionality reduction | principal component analysis | 14 |
| Hidden Markov models | decoding, forward–backward, Baum–Welch; dynamic Bayesian networks | 19, 20 |
| Bayesian learning | Bayesian estimation, MCMC | 21 |
| Neural networks | models, backpropagation | 22 |
| Model selection | bias–variance trade-off, cross-validation, BIC, regularisation | 23 (and 07) |

Modules 01–04 cover the two prerequisite courses of 02-620 (02-601 programming, 02-680 mathematics and statistics),
each tied to the lecture where that mathematics is first used.

### 📘 Lecture notes — 02-620 at a glance: what the course expects and how it works

- **Prerequisites.** 02-601 *Programming for Scientists* and 02-680 *Essential Mathematics and Statistics for
  Scientists* (or equivalent): basic linear algebra, probability and statistics; basic familiarity with Python
  and NumPy — enough to do simple linear algebra in NumPy and to save and open Jupyter notebooks. Recitations
  start with reviews of probability, linear algebra and coding. In this course, Modules 01–04 teach exactly these
  prerequisites.
- **Why Python?** All homework is written in Python. Statisticians traditionally work in **R**, which has a huge
  collection of statistical packages but is awkward for building large software; **MATLAB** is designed for
  matrix computation but is commercial and resource-hungry. **Python** is free and open source, handles matrix
  computation well (NumPy), supports software engineering and deep learning, and has become the main language of
  machine learning and data science.
- **Textbooks (optional).** Murphy, *Machine Learning: a Probabilistic Perspective* (2012) — thick but easy to
  read, with a probabilistic emphasis that links machine learning to statistics; Hastie, Tibshirani & Friedman,
  *The Elements of Statistical Learning* — concise and more demanding, written by statisticians who explain tricky
  topics well; Mitchell, *Machine Learning* (1997) — the oldest, less up to date, but it covers the fundamentals
  very clearly. *The Elements of Statistical Learning* is free from the authors' website, and Murphy's newer
  *Probabilistic Machine Learning* books (2022, 2023) are free online; Murphy (2012) and Mitchell (1997) are in
  most libraries.
- **How learning is assessed.** Written and programming homework, two in-class midterms that are *not*
  cumulative (the second covers only material after the first; one handwritten page of notes is allowed), and a
  team project. In the 2025 offering the weights were homework 50 %, each midterm 12.5 % and the project 25 %
  (check the current syllabus).
- **The course project.** A team of 2–4 students analyses **one data set with three methods of their choice**:
  a proposal (what you plan to do) → peer review of the proposal → a presentation in the final week → a final
  report → peer review of the report. [Module 44](44_capstone_project.ipynb) rehearses such a project from start
  to finish.
- **Academic integrity.** Discussing homework with classmates is welcome, but each student writes their own
  solution in their own words; the university's default penalty for an integrity violation is failing the course.
- **A word of encouragement.** Machine learning is not an easy subject, and finding it challenging is perfectly
  normal. It takes one semester of effort, and what you learn stays with you for many years.

> 🎓 **Exam tip.** Two prerequisites are used from the first week on: probability (conditional probability, Bayes' rule —
    Module 02) and linear algebra (matrices and vectors — Module 03). If either feels shaky, work through those modules
    before the regression lecture.

---
## 2 · Biology became a data science

> 📍 **Where we are.** You can run cells. Now the story told at the start of 02-620: where machine learning came from,
> and how three technological breakthroughs turned biology into a field that *needs* it. Each breakthrough raised a
> question that one part of this course answers.

**Where machine learning comes from.** In the 1950s and 60s, computer scientists dreamt of building machines that
think like humans — which turned out to be far harder than they imagined. Attention shifted to algorithms that *learn
from data*. In the 1970s and 80s, computer scientists and statisticians realised that they were working on the same
thing — **decision trees** (Module 10): computer scientists approached them algorithmically (a tree is a favourite
data structure), statisticians with statistical principles. The two views merged into a new field, **machine
learning**, which sits where **computer science** (algorithms, data structures) meets **statistics** (principles for
reasoning from noisy data). Early machine learning was mostly algorithmic; over the years it adopted more and more
statistical principles — which is why this course is so probabilistic.

**Learning from examples.** Machine learning began with the wish to build machines that learn the way humans do. A
baby learns what a cat is from examples: an adult points at one cat, then another, and says "cat". Machines can learn
from examples too. They need not work like brains — aeroplanes were inspired by birds but fly by a different
mechanism — but the central idea is the same: **learn from examples (data) instead of being programmed with rules.**
The same methods now run in many fields:

* **computer vision** — segment an object (a cat) from the background, or label every region of a scene (dog, cat,
  duck), automatically across a database of millions of images that no human could label;
* **speech recognition** — cut a voice signal into segments and label each segment with a word, then string the words
  into a sentence;
* **text analysis** — extract the topics discussed in a century of newspaper articles and how they evolved; modern
  chatbots such as ChatGPT are a recent breakthrough of the same field;
* **robotics** — a self-driving car combines a *vision* model (what objects are in the camera image?) with a
  *decision* model (slow down, or change lanes and overtake?);
* finance, medicine — and **genomics**, the focus of this course.

People working in all these areas speak the same language: once a problem is formalised as "input X, output Y,
function f" (Section 4), *any* method of this course can be tried on it.

**Biology in the old days.** Until about 2000, a lab typically studied *one gene at a time* — a **candidate gene** with
some evidence linking it to a disease. Mutations in the **HTT** (huntingtin) gene cause Huntington disease; mutations
in **CFTR** cause cystic fibrosis, and CFTR was sequenced in so many patients that many of its disease mutations were
known. But what about the other ~20,000 genes? Studying them one by one was hopelessly slow.

### Breakthrough 1 — the human genome sequence (2001)

In 2001 the first draft of the human genome — **about 3 billion nucleotides** — became public. At the University of
California, Santa Cruz, Jim Kent (then a graduate student) and his advisor David Haussler assembled it and built the
**UCSC Genome Browser** to serve it to everyone; it is still one of the most used resources in genomics. The first
question was: *where are the genes?* (Many expected humans, being more complex than flies, to have many more genes;
that turned out not to be the case.) What a computer receives is just a string of letters. We simulate a short
stretch — 240 letters with a gene structure that we choose, so that we know the truth.

In [ ]:
# ── 2.1 The genome, as a computer sees it: a SIMULATED stretch of DNA with a known gene structure ──
STRUCTURE = [("intergenic", 30), ("exon", 18), ("intron", 25), ("exon", 10), ("intron", 20), ("exon", 15),
             ("intergenic", 35), ("exon", 22), ("intron", 15), ("exon", 14), ("intergenic", 36)]   # (region, length)
BASE_FREQ = {"intergenic": [0.32, 0.18, 0.18, 0.32],    # probabilities of a, c, g, t: AT-rich
             "intron":     [0.30, 0.20, 0.20, 0.30],
             "exon":       [0.18, 0.32, 0.32, 0.18]}    # coding regions are GC-rich


def simulate_dna(structure, rng):
    """Draw the letters of each region from its base frequencies; introns start with 'gt' and end with 'ag'."""
    letters, labels = [], []
    for region, length in structure:
        seq = list(rng.choice(list("acgt"), size=length, p=BASE_FREQ[region]))
        if region == "intron":                  # the splice signals that mark intron boundaries
            seq[:2], seq[-2:] = ["g", "t"], ["a", "g"]
        letters += seq
        labels += [region] * length
    return "".join(letters), np.array(labels)


dna, region_of = simulate_dna(STRUCTURE, np.random.default_rng(SEED))
print(dna[:60], "...")
print("letters:", len(dna))
for base in "acgt":
    print(f"  {base}: {dna.count(base):3d}  ({dna.count(base) / len(dna):.0%})")
print(f"fraction of the whole genome this represents: {len(dna) / genome_bases:.1e}")

🔍 **What just happened?** The DNA is just a string: the cell counted each base and found that this short stretch is
about **one ten-millionth** of one genome. Nothing in the letters says "gene starts here". Biologists knew that genes
have a structure — **exons** (the pieces that code for protein) separated by **introns**, with long **intergenic**
stretches between genes — so the task is to *colour* every letter with its hidden role. Because we simulated the
sequence, we know the true colouring. Run the next cell to see it.

In [ ]:
# ── 2.2 Gene finding = giving every letter a hidden label (the TRUE labels of our simulation) ──
colours = {"intergenic": "tab:blue", "intron": "gold", "exon": "tab:red"}
fig, ax = plt.subplots(figsize=(11, 2.3))
start = 0
for label, length in STRUCTURE:
    ax.add_patch(plt.Rectangle((start, 0), length, 0.8, color=colours[label]))
    start += length
for i, base in enumerate(dna):                       # the observed letters, one per position
    if i % 2 == 0:                                    # every second letter, so that they stay readable
        ax.text(i + 0.5, 1.25, base, ha="center", va="center", fontsize=5.5, family="monospace")
ax.text(-3, 1.25, "observed X\n(letters)", ha="right", va="center", fontsize=8)
ax.text(-3, 0.4, "hidden Y\n(labels)", ha="right", va="center", fontsize=8)
for k, (label, c) in enumerate(colours.items()):
    ax.add_patch(plt.Rectangle((start + 8, 1.4 - 0.5 * k), 6, 0.35, color=c))
    ax.text(start + 16, 1.57 - 0.5 * k, label, va="center", fontsize=8)
ax.set_xlim(-30, start + 40); ax.set_ylim(-0.3, 1.8); ax.axis("off")
ax.set_title(f"SIMULATED gene finding problem: {start} observed letters, each with a hidden label")
plt.tight_layout(); plt.show()

for region in colours:
    in_region = [b for b, r in zip(dna, region_of) if r == region]
    gc = sum(b in "gc" for b in in_region) / len(in_region)
    print(f"{region:10s}: {len(in_region):3d} letters, {gc:.0%} of them g or c")

🔍 **What just happened?** The top row is what the sequencer gives us — the letters (we print every second one). The
coloured bar is what we *want*: the role of each letter. Real gene finders must infer it, but they have clues: in our
simulation (as in many real genomes) **exons are GC-rich** — about two-thirds of their letters are g or c, against
roughly a third elsewhere — and every intron starts with `gt` and ends with `ag`. No single letter decides anything;
the *statistics of stretches of letters* do.

**Borrowing a method from speech recognition.** David Haussler, who had trained in machine learning, saw that this is
the *same problem* as **speech recognition**: a recording is a long signal in which hidden words must be segmented and
labelled ("I love the Santa Cruz beach"); a genome is a very long "sentence" written in a four-letter alphabet (a, c,
g, t) in which hidden exons, introns and intergenic regions must be found. The algorithms developed for speech —
**hidden Markov models**, a kind of **probabilistic graphical model** in which each observed letter is emitted by a
hidden state — could be borrowed to infer the most probable sequence of hidden labels from the observed letters. The
idea was a landmark contribution (Haussler was elected to the US National Academy of Sciences). You will build exactly
such a gene finder in [Module 19](19_hmm_inference.ipynb). The general lesson: *recognise that your biological problem
has been solved in another field, and borrow the method.*

**The data deluge.** Sequencing became cheaper so fast that by 2011 a news feature in *Science* asked whether
computers would "crash genomics": sequencers produced data faster than computing power grew. Without machine learning,
the flood of letters would simply be unreadable.

### Breakthrough 2 — maps of human variation (2001–today)

Any two people differ at millions of positions of their genomes. A position where some people carry one base and
others another — a flip of a **single** nucleotide — is a **single-nucleotide polymorphism (SNP)**. The next cell
simulates the aligned DNA of twelve people, six healthy and six with a disease, at five SNPs.

In [ ]:
# ── 2.3 SNPs: the aligned DNA of 12 SIMULATED people, 6 healthy and 6 with a disease ──
snp_rng = np.random.default_rng(SEED + 1)
reference = snp_rng.choice(list("acgt"), size=36)                 # the common sequence of a short region
status = np.array([0] * 6 + [1] * 6)                               # 0 = healthy, 1 = disease
snp_pos = np.array([4, 11, 19, 27, 32])                            # positions where people can differ
alt = np.array([snp_rng.choice([b for b in "acgt" if b != reference[p]]) for p in snp_pos])   # the other base
p_alt = np.array([[0.2, 0.4, 0.15, 0.5, 0.3],                      # P(other base) in healthy people, per SNP
                  [0.2, 0.4, 0.85, 0.5, 0.3]])                     # ... in patients: only the SNP at 19 differs
seqs = np.tile(reference, (len(status), 1))
for k, p in enumerate(snp_pos):
    carries = snp_rng.random(len(status)) < p_alt[status, k]
    seqs[carries, p] = alt[k]

fig, ax = plt.subplots(figsize=(10, 3.6))
for k, p in enumerate(snp_pos):
    ax.axvspan(p - 0.5, p + 0.5, color="tab:red" if k == 2 else "0.85", alpha=0.35 if k == 2 else 1)
for i, s in enumerate(seqs):
    for j, base in enumerate(s):
        ax.text(j, -i, base, ha="center", va="center", family="monospace", fontsize=8,
                fontweight="bold" if j in snp_pos and base != reference[j] else "normal")
    ax.text(-1.5, -i, ["healthy", "disease"][status[i]], ha="right", va="center", fontsize=7.5,
            color=CLASS_COLORS[status[i]])
ax.axhline(-5.5, color="k", lw=0.8)
ax.set_xlim(-6, 36); ax.set_ylim(-11.8, 1.2); ax.axis("off")
ax.set_title("SIMULATED DNA of 12 people: grey/red columns are SNPs (bold = the less common base); "
             "red = associated with disease", fontsize=9)
plt.tight_layout(); plt.show()

snp_table = pd.DataFrame({"position": snp_pos, "bases": [f"{reference[p]}/{a}" for p, a in zip(snp_pos, alt)]})
other = seqs[:, snp_pos] == alt                                    # True where a person carries the other base
snp_table["freq. of other base"] = other.mean(axis=0).round(2)
snp_table["minor allele freq."] = np.minimum(other.mean(axis=0), 1 - other.mean(axis=0)).round(2)
snp_table["in healthy"] = other[status == 0].mean(axis=0).round(2)
snp_table["in disease"] = other[status == 1].mean(axis=0).round(2)
print(snp_table.to_string(index=False))

🔍 **What just happened?** Each row is one person's version of the same 36-letter region; at 31 positions everyone
agrees, at the five shaded columns people differ — those are the SNPs. The table gives, for each SNP, how often the
less common variant occurs. The **minor allele frequency (MAF)** is the frequency of the *rarer* of the two bases
(never above 0.5). At position 19 the other base is much more frequent among the patients (0.83) than among the
healthy (0.17): that SNP is **associated** with the disease — in our simulation by construction. At positions 4 and 11
the patients also look a little different (0.33 vs 0.00 and 0.17) although the simulation gave both groups the same
probabilities: with six people per group, chance differences are large. Telling real associations from chance ones
across millions of SNPs is the statistical heart of genome-wide association studies (Modules 06–07).

**The maps of human variation.** In 2001 an international SNP map listed **1.42 million** SNPs. In 2005 **HapMap
Phase I** genotyped at least one *common* SNP (MAF > 5 %) every 5,000 bases — about **1 million** SNPs — in **270
people** from four populations (30 Yoruba parent–child trios from Ibadan, Nigeria; 30 trios of European ancestry from
Utah; 45 unrelated Han Chinese from Beijing; 45 unrelated Japanese from Tokyo). The **1000 Genomes Project** (2010)
sequenced whole genomes from many populations; as costs kept falling, hospitals began sequencing newborns (BabySeq);
and **UK Biobank** follows **500,000** volunteers with genotypes and health records to improve the prevention,
diagnosis and treatment of cancer, heart disease, stroke, diabetes, dementia and other illnesses (it was used, for
example, to study COVID-19 outcomes). Two questions arose:

* *Which variants cause disease?* — comparing the genomes of healthy and affected people is a **regression or
  classification** problem: genome-wide association studies, Modules 06–11. Finding causal variants gives clues for
  drug and treatment design.
* *What does a genome say about ancestry?* — companies such as 23andMe infer ancestry from genotypes. Novembre *et al.*
  (2008) analysed genotypes at **197,146** SNPs in **1,387** Europeans and compressed each person to **2 numbers** by
  **principal component analysis**; plotted, the people formed a map of Europe. That is **dimensionality reduction**
  (Module 14); Section 5 simulates such a map.

### Breakthrough 3 — measuring all genes at once (1999–today)

Knowing that a mutation causes a disease does not explain *how*: to treat it we must understand the **biological
mechanism** the mutation disturbs. DNA tells you what *could* happen; to see mechanism you must see which genes are
**expressed**. **Microarrays** measure the expression of thousands of genes per sample, one spot per gene on a small
chip — leading to Golub's leukaemia study (**classification and clustering**, Modules 08–14) and to the learning of
**gene regulatory networks** from the expression of hundreds of individuals (**probabilistic graphical models**,
Modules 15–18). Today **RNA-seq** counts the messenger RNAs directly and more accurately, and **single-cell RNA-seq**
does so in each of tens of thousands of individual cells (Level 2, Modules 30–34).

How big are these data? Run the next cell.

In [ ]:
# ── 2.4 How many numbers does each breakthrough produce? (sizes of the original studies) ──
studies = pd.DataFrame([
    ("Human genome draft (2001)", "1 genome × 3 billion bases", 3e9, "where are the genes? → HMMs", "19–20"),
    ("HapMap Phase I (2005)", "270 people × ≈ 1 million SNPs", 270 * 1.0e6, "which variants matter? → regression, classification", "06–11"),
    ("Novembre et al. (2008)", "1,387 people × 197,146 SNPs", 1387 * 197_146, "ancestry → PCA", "14"),
    ("Golub et al. (1999)", "72 patients × 7,129 probes", 72 * 7129, "leukaemia subtypes → classification, clustering", "08–14"),
], columns=["study", "shape", "numbers", "question → method", "modules"])
pd.set_option("display.max_colwidth", 60)
print(studies.to_string(index=False, formatters={"numbers": "{:.2e}".format}))
print("\nUK Biobank: 500,000 participants with genotypes and health records.")

fig, ax = plt.subplots(figsize=(8, 2.8))
ax.barh(studies["study"], studies["numbers"], color=["tab:blue", "tab:orange", "tab:green", "tab:red"])
ax.set_xscale("log"); ax.set_xlim(1e5, 1e10); ax.invert_yaxis()
ax.set_xlabel("number of measurements in the data set (log scale: each tick = 10× more)")
ax.set_title("Genome-scale biology produces far more numbers than anyone can read")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The data sets range from about **5 × 10⁵** numbers (Golub: 72 × 7,129 = 513,288) to
**3 × 10⁹** letters (one genome); HapMap and the European study each hold a few hundred million genotypes. On a
**logarithmic axis** each tick is ten times the previous one — the only way to fit such different sizes in one picture.
No human can read 300 million genotypes; an algorithm can.

⚠️ **Big is not the same as "many samples".** Look at the *shape*: Golub has only **72 patients** but **7,129
measurements per patient**; Novembre has 1,387 people but 197,146 SNPs each. Biological data are usually **wide**: far
more features than samples. This single fact causes many of the difficulties in this course — least squares breaking
down (Module 07), choosing which genes to trust (Modules 07, 23), and the danger of finding patterns by chance (as at
positions 4 and 11 of cell 2.3).

### 📘 Lecture notes — machine learning in science and the three breakthroughs

- **Machine learning** = algorithms that **learn from examples (data)**; it grew where computer science
  (algorithms) met statistics (reasoning under noise) — the two communities converged in the 1970s–80s on decision
  trees — and became steadily more statistical. The same methods serve vision (segmenting and labelling objects),
  speech (segmenting a signal into words), text (topics), robotics (perception + decisions) and biology.
- **Old biology:** one candidate gene at a time (HTT → Huntington disease, CFTR → cystic fibrosis).
- **Breakthrough 1 — the genome sequence (2001, ≈ 3 × 10⁹ bases; UCSC Genome Browser).** Question: *where are the
  genes?* Genes = exons separated by introns, between intergenic stretches; gene finding labels **every letter**.
  Haussler's insight: this is speech recognition with a 4-letter alphabet → **hidden Markov models / probabilistic
  graphical models** (Modules 19–20).
- **Breakthrough 2 — maps of variation.** A **SNP** is a single-nucleotide difference between people. SNP map
  (2001, 1.42 M SNPs); HapMap I (2005: ≈ 1 M common SNPs, about one every 5 kb, in 270 people from four populations);
  1000 Genomes (2010); BabySeq; UK Biobank (500,000 people). Questions: *which variants cause disease?* →
  **regression / classification** (06–11); *ancestry and population structure* → **PCA** (14; 197,146 SNPs in
  1,387 Europeans reduced to 2 numbers per person form a map of Europe).
- **Breakthrough 3 — expression of all genes.** From mutation to *mechanism*: microarrays → RNA-seq →
  single-cell RNA-seq. Golub's ALL/AML study → **classification and clustering** (08–14); gene regulatory networks
  → **probabilistic graphical models** (15–18).

**Minor allele frequency.** If a fraction $p$ of the chromosomes in a population carry one base at a SNP and $1-p$
carry the other,

$$\text{MAF} = \min(p,\; 1 - p)$$

*In words:* the MAF is the frequency of the rarer of the two variants, so it is always between 0 and 0.5.
*Worked example:* if 30 of 200 chromosomes carry T and 170 carry A, then $p_T = 0.15$ and MAF = 0.15 — a
**common** SNP by HapMap's definition (MAF > 5 %).

**Common confusions.** A SNP is a *position* that varies, not a mutation in one person. "Big data" in biology
usually means many *features* per sample (J ≫ N), not many samples.

> 🎓 **Exam tip.** For each breakthrough be able to name the biological question *and* the family of machine-learning
    methods attached to it: genome → gene finding → HMMs/graphical models; variation → disease genes by
    regression/classification and ancestry by PCA; expression → subtypes by classification/clustering and gene
    networks by graphical models.

---
## 3 · Data as tables

> 📍 **Where we are.** Section 2 showed that biology produces huge collections of numbers. Every method in this course
> starts by arranging them as a **table**. Here we generate one: leukaemia expression data shaped like Golub's.

Almost every data set in 02-620 is a table (in mathematics: a **matrix**) called **X**:

* each **row** is one **sample** — a patient, an individual, a cell. There are **N** rows;
* each **column** is one **feature** — a gene's expression, a SNP's genotype, a protein's level. There are **J** columns;
* the number in row *i*, column *j* is the measurement of feature *j* in sample *i*.

If we also know an outcome for each sample (ALL or AML, healthy or sick, a blood-pressure value), we store it in a
separate column **y** with one entry per row. This **N × J** convention (samples in rows) is the one used throughout
02-620 (from the regression lecture on) and throughout this course.

⚠️ **Biologists often draw it the other way round.** The famous heatmap of Golub *et al.* has *genes as rows* and
*patients as columns*, as most expression papers do. Same data, transposed. Always check which way a table is oriented
before you compute anything.

> 🧪 **Why simulated data?** This course stores no data sets in its repository: every data set is generated by code in
> the notebook that uses it, so each notebook runs on its own anywhere. Simulation has a second, scientific advantage:
> **we know the truth.** The simulator records which genes truly differ between ALL and AML and each patient's
> subtype, so we can check whether a method found them — something real data never allow. The price: our numbers
> are *not* the numbers of the original study, and we never claim they are.

In [ ]:
# ── 3.1 Generate the leukaemia data (SIMULATED, shaped like Golub et al. 1999) ──
leuk = simulate_leukemia_expression()   # dict: X, y, labels, subtype, genes, is_train, sample_id, true_effect, source
X, y, genes = leuk["X"], leuk["y"], leuk["genes"]
print("Source:", leuk["source"])
print("X has shape", X.shape, "→ N =", X.shape[0], "patients (rows), J =", X.shape[1], "genes (columns)")
names_, counts_ = np.unique(leuk["labels"], return_counts=True)
print("Classes:", {str(k): int(v) for k, v in zip(names_, counts_)}, "(y = 0 for ALL, 1 for AML)")
names_, counts_ = np.unique(leuk["subtype"], return_counts=True)
print("Subtypes:", {str(k): int(v) for k, v in zip(names_, counts_)})
print("Original split:", leuk["is_train"].sum(), "training and", (~leuk["is_train"]).sum(), "test patients")
print(f"Values range from {X.min():.2f} to {X.max():.2f} (log10 of expression, i.e. {10 ** X.min():.0f} to "
      f"{10 ** X.max():,.0f}); {np.mean(X == X.min()):.0%} of all entries sit at the lower limit")

🔍 **What just happened?** The simulator produced **72 patients (47 ALL, 25 AML) × 3,571 genes** — the shape of the
original data after the standard preprocessing of Dudoit, Fridlyand & Speed (2002): the arrays measured 7,129 probes;
values were clipped to the range 100–16,000, genes that barely vary were removed (leaving 3,571), and the numbers were
converted to **log10** — so 2.0 means 100 units and 4.2 means about 16,000. The logarithm turns "twice as much" into
"+0.3" wherever you are on the scale. About **23 %** of all entries sit exactly at the floor of 2.0: the gene was
below the detection limit in that patient. As in the original study, the ALL patients come in two lineages — **38
B-cell and 9 T-cell ALL** — and **38 patients are used to train**, **34 are kept to test**.

Let us peek at a few numbers. Some genes carry the names of classic leukaemia markers; we look up four of them.

In [ ]:
# ── 3.2 A peek at the table: 6 patients × 4 named marker genes ──
col = {g: j for j, g in enumerate(genes)}                       # gene symbol → column number
print(f"{len(LEUKEMIA_MARKERS)} of the {len(genes):,} genes are named after classic leukaemia markers, e.g.:")
show = ["ZYX", "CD33", "MYB", "CD79A"]
for g in show:
    print(f"  {g:6s} {LEUKEMIA_MARKERS[g][1]:12s} — high in {LEUKEMIA_MARKERS[g][0]}")
rows = np.r_[0:3, np.flatnonzero(y == 1)[:3]]                   # three ALL and three AML patients
peek = pd.DataFrame(X[np.ix_(rows, [col[g] for g in show])], columns=show,
                    index=[f"patient {leuk['sample_id'][r]} ({leuk['subtype'][r]})" for r in rows])
print(peek.round(2))

🔍 **What just happened?** In this little 6 × 4 slice, **CD33** is higher (larger log10 values) in all three AML
patients (3.07–3.27 vs 2.58–2.89) and **CD79A** is higher in all three ALL patients (3.76–4.20 vs 2.85–3.10; 4.20 is
the ceiling of the scale). **MYB** points the same way *on average* but nearly overlaps (AML patient 18 has 2.54, ALL
patient 1 has 2.56), and two AML patients sit at or near the floor (2.00, 2.05). **ZYX** — a strong AML marker in the
original study — does not separate these six patients at all: ALL patients 2 and 3 (4.12, 4.06) exceed AML patient 14
(3.90). The directions make biological sense: CD33 is a surface marker of myeloid cells, CD79A (also called MB-1) is
part of the B-cell receptor, and c-Myb is a transcription factor active in immature lymphoid cells. But one gene and a
handful of patients cannot settle anything; we need to look at *all* patients and *many* genes.

### 3.3 Which genes differ most between ALL and AML?

Golub *et al.* ranked every gene by a **signal-to-noise score** computed on the training patients only:

$$P(g) = \frac{\mu_{\text{ALL}}(g) - \mu_{\text{AML}}(g)}{\sigma_{\text{ALL}}(g) + \sigma_{\text{AML}}(g)}$$

*In words:* for gene $g$, take the difference between its average expression in ALL and in AML patients ($\mu$, the
Greek letter "mu", stands for an average), and divide by how much it varies within each class ($\sigma$, "sigma", is
the standard deviation — the typical spread). A large positive score means "reliably higher in ALL", a large negative
score "reliably higher in AML". Dividing by the spread rewards genes whose difference is large *compared with the
patient-to-patient variation*. (You will meet means and standard deviations properly in Module 02.)

We compute the score for all 3,571 genes and keep the 25 most ALL-like and the 25 most AML-like, as Golub *et al.*
did for their famous figure.

In [ ]:
# ── 3.3 Rank all genes by Golub's signal-to-noise score (training patients only) ──
def signal_to_noise(values: np.ndarray, labels: np.ndarray) -> np.ndarray:
    """Golub's score (mean_ALL − mean_AML) / (sd_ALL + sd_AML) for every column; 0 where both sds are 0."""
    a, b = values[labels == 0], values[labels == 1]
    spread = a.std(axis=0, ddof=1) + b.std(axis=0, ddof=1)
    return np.where(spread > 0, (a.mean(axis=0) - b.mean(axis=0)) / np.where(spread > 0, spread, 1.0), 0.0)


def gene_label(g: str) -> str:
    """'CST3 · cystatin C' for a named marker, the plain ID ('G0872') otherwise."""
    return f"{g} · {LEUKEMIA_MARKERS[g][1]}" if g in LEUKEMIA_MARKERS else g


train = leuk["is_train"]
score = signal_to_noise(X[train], y[train])            # on the log10 scale of the data
order = np.argsort(score)                               # from most AML-like (negative) to most ALL-like
top_ALL, top_AML = order[::-1][:25], order[:25]
chosen = np.r_[top_ALL, top_AML]
print("Most ALL-like:", [gene_label(g) for g in genes[top_ALL[:5]]])
print("Most AML-like:", [gene_label(g) for g in genes[top_AML[:5]]])

🔍 **What just happened?** The five most ALL-like genes are TCF3 (E2A), TOP2B, CD24, LMO2 and STMN1 (Op18); the five
most AML-like are CST3 (cystatin C), ELANE, AZU1, NFKBIA and MPO (myeloperoxidase). All carry marker names: the
ranking found the genes the simulator made different. Because the data are simulated, we can check the whole list
against the truth.

In [ ]:
# ── 3.4 🧪 Ground-truth check: did the ranking find genes that truly differ? ──
truth = leuk["true_effect"]                             # simulated AML − ALL difference of every gene (0 = none)
right_direction = np.r_[truth[top_ALL] < 0, truth[top_AML] > 0]
named = np.isin(genes[chosen], list(LEUKEMIA_MARKERS))
print(f"genes with a true difference in the whole table: {np.mean(truth != 0):.0%} of {len(genes):,}")
print(f"among our top 50: {right_direction.sum()} truly differ in the direction the score says; "
      f"{named.sum()} are named markers ({named[:25].sum()} ALL-high, {named[25:].sum()} AML-high)")

🔍 **What just happened?** In the whole table only **26 %** of the genes truly differ between the classes, yet **all 50**
of our top genes do, and each in the direction its score claims. **32** of them are named markers (the genes with the
largest simulated differences); the other 18 have smaller true differences that happened to be very consistent in the
38 training patients. With real data this check is impossible: nobody knows the true list.

Now the classic picture — for **all 72 patients**, including the 34 test patients the ranking never saw.

In [ ]:
# ── 3.5 Heatmap: 50 genes × 72 patients, patients sorted B-ALL | T-ALL | AML ──
subtype = leuk["subtype"]
patients = np.r_[np.flatnonzero(subtype == "B-ALL"), np.flatnonzero(subtype == "T-ALL"), np.flatnonzero(subtype == "AML")]
Z_genes = standardize(X[:, chosen])[0]                                     # each gene: mean 0, sd 1 across patients
n_B, n_T = (subtype == "B-ALL").sum(), (subtype == "T-ALL").sum()
fig, ax = plt.subplots(figsize=(9.5, 7.2))
im = ax.imshow(Z_genes[patients].T, aspect="auto", cmap="RdBu_r", vmin=-2.5, vmax=2.5, interpolation="nearest")
ax.axvline(n_B + n_T - 0.5, color="k", lw=1.5); ax.axvline(n_B - 0.5, color="k", lw=0.8, ls="--")
ax.axhline(24.5, color="k", lw=1.5)
ax.set_yticks(range(50)); ax.set_yticklabels([gene_label(g) for g in genes[chosen]], fontsize=5.5)
ax.set_xticks([n_B / 2, n_B + n_T / 2, n_B + n_T + (y == 1).sum() / 2])
ax.set_xticklabels([f"B-ALL ({n_B})", f"T-ALL ({n_T})", f"AML ({(y == 1).sum()})"])
ax.set_xlabel("patients (columns, as in the classic figure)"); ax.set_ylabel("genes (rows)")
fig.colorbar(im, ax=ax, fraction=0.025, label="expression (z-score: sd above/below the gene's mean)")
ax.set_title(f"{leuk['source']}:\ntop-25 ALL-high (upper) and top-25 AML-high (lower) genes")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Red means "higher than this gene's average", blue "lower". The upper block (ALL-high genes)
is red on the left and blue on the right; the lower block (AML-high genes) is the mirror image — the checkerboard of
Golub's original figure, in which the authors saw that the two leukaemias have distinct expression programmes. The
simulated pattern is crisper than the real one (real data are noisier), but still not perfect: here and there an ALL
patient has an AML gene switched on, or an AML patient an ALL gene. No single gene separates the classes flawlessly, but **many genes
together** do. Combining many weak clues into one decision is precisely what a classifier does.

✏️ **Your turn.** In cell 3.3, replace `X[train]` by `10 ** X[train]` (rank on the original, un-logged scale, as Golub
*et al.* did) and re-run cells 3.3–3.5. How many named markers and how many truly different genes are in the top 50
now? (We found 27 named markers and 49 true genes — a reminder that "preprocessing choices" change which genes look
important.) Change it back afterwards.

---
## 4 · Supervised learning: learning a function from labelled examples

> 📍 **Where we are.** We have a table X (expression) and a label y (ALL/AML) for every patient. Now the first big
> family of machine-learning problems: learn to predict y from X. Almost every algorithm in this course is either
> *supervised* or *unsupervised*, so this distinction is the first question to ask about any learning problem.

**Supervised learning** means learning to perform the task *"given input X, decide output Y"* — learning a
**function** $f: X \to Y$ that maps an input to an output. The data come as **pairs** (X, Y); Y is the *teacher* that
supplies the correct answer, just as an adult tells a baby "that's a cat". Examples: blood-test results → healthy
(Y = 0) or heart disease (Y = 1); an image → is there an object in it? is it a cat? For us: a patient's expression
profile → ALL (Y = 0) or AML (Y = 1). (The teacher need not be human: in *teacher–student learning*, a model learns
from the predictions of a more powerful model.) The paired data are called the **training data**; the output is also
called the **label** or **target**; finding f is called **training** or **learning**. Humans can learn "raccoon" from a
single picture; machines usually need many examples, so the **sample size** matters.

The workflow has three stages:

1. **Choose** a form for $f$ — a line? a polynomial? a tree? a graph? a network?
2. **Learn** it from the training data. There is a trade-off: a very powerful form may be hard to learn, while a
   simple, even naïve-looking one that can be learned quickly and reliably is often preferable.
3. **Evaluate** it on *previously unseen* inputs — like an exam whose answers you have not seen: you give your best
   guess for Y, then the teacher brings out the true labels and grades you (for example, the fraction of predictions
   that match the truth).

Two kinds of supervised problems:

* **Regression** — Y is a **continuous** number: a blood-pressure value, a gene's expression level, the PSA level of a
  prostate-cancer patient (Module 06).
* **Classification** — Y is **discrete** (a category): ALL vs AML, healthy vs diseased, cancer subtype (Modules 08–11).

**Modelling uncertainty.** Instead of a bare answer, we would like $f$ to say *how sure* it is. 02-620 therefore
formulates learning $f$ as learning a **conditional probability** $P(Y \mid X)$ — read "the probability of Y given
X". For a cat picture, $P(\text{cat} \mid X) = 0.9$ and $P(\text{dog} \mid X) = 0.1$ means "fairly sure it is a cat";
$0.5$ and $0.5$ means "no idea". The probabilities of all possible answers for the same X add up to 1. For a doctor
this is crucial: a confident prediction can be trusted, an uncertain one should be sent for more tests or reviewed by
a human. The next cell draws the whole workflow.

In [ ]:
# ── 4.1 The supervised-learning workflow as a picture ──
def box(ax, x, y, w, h, text, colour):
    """A rounded, lightly coloured box with centred text."""
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.03", fc=colour, ec="0.3", alpha=0.3))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=8)


def arrow(ax, start, end, text=""):
    """An arrow from start to end with an optional label above its middle."""
    ax.annotate("", xy=end, xytext=start, arrowprops=dict(arrowstyle="-|>", color="0.3", lw=1.2))
    if text:
        ax.text((start[0] + end[0]) / 2, (start[1] + end[1]) / 2 + 0.12, text, ha="center", va="bottom",
                fontsize=7.5, color="0.25")


fig, ax = plt.subplots(figsize=(11, 3.4))
box(ax, 0.0, 1.0, 2.4, 1.8, "TRAINING DATA\npairs (X, Y)\n\nX: expression profile\nY: ALL or AML\n(Y is the teacher)",
    "tab:blue")
arrow(ax, (2.5, 1.9), (3.3, 1.9), "1 · choose\n2 · learn")
box(ax, 3.4, 1.4, 2.0, 1.0, "learned function\nf: X → Y\n(modelled as P(Y | X))", "tab:orange")
box(ax, 3.4, -0.1, 2.0, 0.8, "NEW input X\n(an unseen patient)", "tab:green")
arrow(ax, (4.4, 0.75), (4.4, 1.35))
arrow(ax, (5.5, 1.9), (6.2, 1.9), "predict")
box(ax, 6.3, 1.1, 2.4, 1.6, "What is Y?\nP(AML | X) = 0.9, P(ALL | X) = 0.1\n→ confident\n"
                            "P(AML | X) = 0.5, P(ALL | X) = 0.5\n→ uncertain", "tab:purple")
arrow(ax, (8.8, 1.9), (9.3, 1.9), "3 · evaluate")
box(ax, 9.4, 1.1, 1.7, 1.6, "compare with\nthe TRUE label\n(an exam graded\nafterwards)", "tab:red")
ax.set_xlim(-0.1, 11.2); ax.set_ylim(-0.25, 3.0); ax.axis("off")
ax.set_title("Supervised learning: learn f from labelled pairs, apply it to unseen inputs, grade the predictions")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The picture is the plan for the rest of this section. Let us build the simplest classifier
we can think of, using just **two genes**: ZYX (zyxin, AML-high) and MYB (c-Myb, ALL-high) — two of the genes that
distinguished the leukaemias in the original study.

In [ ]:
# ── 4.2 Two genes, training and test patients ──
GENE_A, GENE_B = "ZYX", "MYB"                         # ✏️ try other names from LEUKEMIA_MARKERS, e.g. "CD33", "TCF3"
two = X[:, [col[GENE_A], col[GENE_B]]]
test = ~train
Z_train, mu_train, sd_train = standardize(two[train])          # z-scores using the TRAINING mean and sd only
Z_test = standardize(two[test], mu_train, sd_train)[0]         # the test patients are scaled the same way
print(f"training patients: {train.sum()} ({(y[train] == 0).sum()} ALL, {(y[train] == 1).sum()} AML); "
      f"test patients: {test.sum()} ({(y[test] == 0).sum()} ALL, {(y[test] == 1).sum()} AML)")

fig, ax = plt.subplots(figsize=(4.6, 4))
for c, name in enumerate(["ALL", "AML"]):
    m = y[train] == c
    ax.scatter(Z_train[m, 0], Z_train[m, 1], s=30, color=CLASS_COLORS[c], edgecolor="k", lw=0.3, label=f"{name} (training)")
ax.set_xlabel(f"{GENE_A} expression (z-score)"); ax.set_ylabel(f"{GENE_B} expression (z-score)")
ax.set_title("SIMULATED: 38 training patients, 2 genes"); ax.legend()
plt.tight_layout(); plt.show()

🔍 **What just happened?** Each dot is one training patient, placed by two numbers. We converted each gene to
**z-scores** (how many standard deviations above or below the training average), so that both axes have the same
scale. The AML patients (red) sit mostly at the **lower right** (higher ZYX, lower MYB) and the ALL patients (blue) at
the **upper left**, but ZYX alone separates them poorly: ALL patients spread across its whole range. Several AML
patients line up in a vertical column at the right edge — their ZYX values hit the ceiling of the measurement scale
(4.2), so they share the same highest value — and two sit at the bottom, at the MYB detection floor (2.0). Floors and
ceilings like these are typical of microarray data.

**A nearest-centroid classifier.** Compute the average position of the ALL patients and of the AML patients — the two
**centroids** (centres of mass). A new patient is assigned to the class whose centroid is **closer**. To turn distances
into probabilities we use

$$P(\text{AML} \mid x) = \frac{e^{-d_{\text{AML}}^2/2}}{e^{-d_{\text{ALL}}^2/2} + e^{-d_{\text{AML}}^2/2}}$$

*In words:* $d_{\text{AML}}$ is the distance from the patient's point $x$ to the AML centroid (and $d_{\text{ALL}}$ to
the ALL centroid); $e^{-d^2/2}$ is large (close to 1) for a nearby centroid and tiny for a distant one; dividing by the
sum makes the two probabilities add up to 1. Equal distances give 0.5/0.5 — maximal uncertainty. (Why exactly this
formula? It is what a simple probability model of each class implies; Modules 08 and 09 derive formulas like it.
Today, just use it.)

In [ ]:
# ── 4.3 Train the nearest-centroid classifier and test it on the 34 unseen patients ──
def fit_centroids(Z: np.ndarray, labels: np.ndarray) -> np.ndarray:
    """Return a (2 × n_features) array: the average point of class 0 (ALL) and of class 1 (AML)."""
    return np.vstack([Z[labels == 0].mean(axis=0), Z[labels == 1].mean(axis=0)])


def centroid_probabilities(Z: np.ndarray, centroids: np.ndarray) -> np.ndarray:
    """P(class | x) ∝ exp(−distance² / 2) to each centroid; each row sums to 1."""
    d2 = ((Z[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)    # squared distance to each centroid
    w = np.exp(-(d2 - d2.min(axis=1, keepdims=True)) / 2)                # subtract the minimum for numerical safety
    return w / w.sum(axis=1, keepdims=True)


centroids = fit_centroids(Z_train, y[train])
p_test = centroid_probabilities(Z_test, centroids)[:, 1]                # P(AML | x) for each test patient
pred_test = (p_test > 0.5).astype(int)
pred_train = (centroid_probabilities(Z_train, centroids)[:, 1] > 0.5).astype(int)
print("centroids (z-scores):", dict(ALL=centroids[0].round(2), AML=centroids[1].round(2)))
print(f"training accuracy: {accuracy(y[train], pred_train):.3f}")
print(f"test accuracy:     {accuracy(y[test], pred_test):.3f}  "
      f"({int(np.sum(pred_test == y[test]))} of {test.sum()} unseen patients correct)")
print("test confusion counts (positive = AML):", confusion_counts(y[test], pred_test))

🔍 **What just happened?** "Training" here means computing two averages — the whole learned model is four numbers.
Yet on the **34 patients it never saw**, the classifier gets **32 right (94 %)**; on the training patients it scores
0.974. The confusion counts tell us *which* mistakes were made: an AML patient called ALL is a **false negative** (FN)
and an ALL patient called AML a **false positive** (FP), with "positive" meaning AML. Here every ALL patient was
recognised (TN = 20, FP = 0) and both errors are AML patients predicted to be ALL (FN = 2) — in medicine, *which*
error you make matters as much as how many: a missed AML means the wrong treatment.

It is normal for test accuracy to differ from training accuracy: the model was fitted to the training patients.
**Only the test score tells us how the classifier would do on new patients** — which is why 02-620 insists on
evaluating on unseen data.

In [ ]:
# ── 4.4 What the classifier decides everywhere, and how sure it is ──
def predict_label(points: np.ndarray) -> np.ndarray:
    """Class (0 = ALL, 1 = AML) of the nearer centroid, for an (M × 2) array of z-scored points."""
    return (centroid_probabilities(points, centroids)[:, 1] > 0.5).astype(int)


fig, (a1, a2) = plt.subplots(1, 2, figsize=(10.5, 4))
plot_decision_regions(predict_label, Z_train, y[train], ax=a1, class_names=["ALL (train)", "AML (train)"])
for c in (0, 1):
    m = y[test] == c
    a1.scatter(Z_test[m, 0], Z_test[m, 1], marker="s", s=34, facecolor="none", edgecolor=CLASS_COLORS[c], lw=1.4,
               label=["ALL (test)", "AML (test)"][c])
a1.scatter(*centroids.T, marker="X", s=140, color="k", label="centroids")
a1.set_xlabel(f"{GENE_A} (z-score)"); a1.set_ylabel(f"{GENE_B} (z-score)"); a1.legend(fontsize=6.5, loc="lower left")
a1.set_title("decision regions (shaded) learned from the 38 training patients")

order_p = np.argsort(p_test)
a2.axhspan(0.3, 0.7, color="0.9", label="uncertain (0.3–0.7)")
a2.scatter(range(test.sum()), p_test[order_p], c=[CLASS_COLORS[c] for c in y[test][order_p]], edgecolor="k", lw=0.3, s=30)
a2.axhline(0.5, color="k", ls="--", lw=1)
a2.set_xlabel("test patients, sorted by predicted probability"); a2.set_ylabel("P(AML | expression)")
a2.set_title("blue = truly ALL, red = truly AML"); a2.legend(loc="upper left", fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** *Left:* the classifier splits the plane by a straight line, halfway between the two
centroids (black crosses); every point on the red side is called AML. Open squares are the test patients: most fall on
the correct side, but two red squares (AML patients) sit on the blue side. *Right:* the predicted probability for
each test patient. Most are confidently near 0 (ALL) or 1 (AML), like "cat 0.9 / dog 0.1"; two fall in the grey
**uncertain band**, like "0.5 / 0.5". Look at the two red dots below the dashed line: those are the errors.

In [ ]:
# ── 4.5 The most uncertain test patients ──
results = pd.DataFrame({"patient": leuk["sample_id"][test], "true class": leuk["labels"][test],
                        "P(AML|x)": p_test.round(3), "prediction": np.where(pred_test == 1, "AML", "ALL")})
results["|P − 0.5|"] = (results["P(AML|x)"] - 0.5).abs().round(3)
print("The six test patients closest to 0.5:")
print(results.sort_values("|P − 0.5|").head(6).to_string(index=False))
print("\nThe misclassified test patients:")
print(results[results["true class"] != results["prediction"]].to_string(index=False))
print("\n🧪 AML patients the simulator gave a weak (atypical) signature:", leuk["sample_id"][leuk["atypical"]].tolist())

🔍 **What just happened?** The first table lists the test patients whose probability is closest to 0.5 — the ones a
cautious doctor would send for more tests. Patient 40 (ALL, P = 0.43) was classified correctly, but only just. Patient
67 (AML, P = 0.33) is one of the two errors, and it sits in the uncertain band: there the model *knew* it was unsure.
The other error is more sobering: patient 47 is AML, yet the model gave P(AML | x) = 0.18, i.e. 82 % for ALL — it was
**wrong with fair confidence**. Its ZYX and MYB values simply look ALL-like. The 🧪 line explains both errors: 47 and
67 are exactly the two AML patients to whom the simulator gave a weak AML signature — no classifier built on these
genes could be expected to recognise them easily. A probability expresses uncertainty only
*within the model*; a model that sees only two genes cannot know what it is missing. This is why 02-620 models
$P(Y \mid X)$ rather than a bare $f(X)$ — and why the *model* behind the probability must be good.

✏️ **Your turn.** In cell 4.2 set `GENE_A, GENE_B = "CD33", "TCF3"` and re-run cells 4.2–4.5. What is the test accuracy
now, and how many patients are uncertain? Then try `"CD79A"` instead of `"TCF3"`. Which pair works best — and why would
it be **cheating** to pick the pair by looking at the *test* accuracy? (Module 23 calls this *selection bias*: once you
use the test patients to make choices, they no longer measure performance on new patients.)

⚠️ **Two cautions.** (1) A probability from a toy model is not a calibrated clinical risk: our 0.9 means "closer to the
AML centroid", nothing more. (2) On the real data, Golub *et al.* used 50 genes and a weighted vote and reported that 29
of the 34 test patients were predicted with high confidence, all of them correctly; our two-gene toy on simulated data
only illustrates the idea.

### 📘 Lecture notes — supervised learning

**Definition.** Supervised learning learns to perform the task "given input X, decide output Y": a function

$$f: X \to Y$$

*In words:* f takes an input (an expression profile, an image, a blood test) and returns an output (a diagnosis,
a label). It is learned from **training data** — pairs (X, Y) in which Y, the **label** or **target**, is the
correct answer supplied by a "teacher".

**Three stages.** (1) Choose the *form* of f (linear, polynomial, tree, graph, network…). (2) *Learn* f from the
training pairs — a powerful form that is hard to learn is not always better than a simple form that is easy to
learn. (3) *Evaluate* f on previously unseen inputs: predict, then compare with the true labels (e.g. the fraction
correct).

**Two kinds.** Y continuous → **regression**; Y discrete → **classification**. The type of **Y** decides, not
the type of X.

**Modelling uncertainty.** Learning f is formulated as learning the conditional distribution $P(Y \mid X)$, whose
values over all possible answers sum to one:

$$\sum_{y} P(Y = y \mid X) = 1$$

*In words:* for one and the same input X, the probabilities of all possible outputs add up to 1 — the model
spreads one unit of belief over the answers.
*Worked example:* P(cat | X) = 0.9 and P(dog | X) = 0.1 (sum 1) means "confidently a cat"; 0.5 and 0.5 means
"completely uncertain". In Section 4 the two-gene classifier gave 32 of 34 test patients the right label; one
error was flagged as uncertain (P(AML) = 0.33), the other was wrong with fair confidence (P(AML) = 0.18).

**Why probabilities?** A doctor (or any user) can decide when to trust the machine and when a human must look
again. A probability is only as good as the model that produced it.

> 🎓 **Exam tip.** Be able to classify a problem as regression or classification from the type of Y alone (not X), and to
    explain why P(Y|X) is more useful than a hard label. Remember: probabilities of all classes given the *same* X sum
    to 1 — P(cat|X) + P(dog|X) = 1, not P(cat|X) + P(cat|X′).

---
## 5 · Unsupervised learning: structure without labels

> 📍 **Where we are.** Section 4 used the labels. But the title of Golub's paper also promises *class discovery*: could
> we have found the two leukaemias without being told? This is the second family of problems.

**Unsupervised learning** means *"find interesting patterns in X"* — there is **no Y**, no teacher. Examples: group a
set of images into horses, cats, dogs and humans, or a century of newspaper articles into topics (politics, business,
entertainment), without anyone labelling them. Why bother? Labels are expensive (someone must provide them), while
unlabelled data are cheap and plentiful — the internet holds billions of unlabelled images and pages. But unsupervised
learning is **much harder** than supervised learning, for computers and humans alike: without correct answers it is not
even obvious how to measure success.

In probability language, supervised learning models $P(Y \mid X)$, while unsupervised learning models the
**distribution of the data themselves**, $P(X)$ — which values of X are common, which are rare, which groups exist.
Learning $P(X)$ is usually harder than learning $P(Y \mid X)$, because the distribution must capture the differences
among all the samples. In practice we bring a "gut feeling" about the patterns we expect (groups? a few main
directions? a network?) and build it into the choice of probability model — assumptions that make $P(X)$ learnable.

Below, a method called **principal component analysis (PCA)** squeezes each patient's 3,571 numbers into just **2**
numbers that keep as much of the patient-to-patient variation as possible — exactly what Novembre *et al.* did with
197,146 SNPs. Treat PCA as a black box today; [Module 14](14_pca_dimensionality_reduction.ipynb) opens it. Crucially,
**PCA never sees the labels**: we colour the points only *afterwards*.

In [ ]:
# ── 5.1 A 2-D map of all 72 patients from all 3,571 genes — computed WITHOUT the labels ──
def pca_scores(data: np.ndarray, k: int = 2):
    """Black box until Module 14: coordinates of each row on the k main axes of variation, and the share of
    the total variance that each axis captures."""
    centred = data - data.mean(axis=0)
    U, S, Vt = np.linalg.svd(centred, full_matrices=False)
    # an axis has no natural +/− direction: fix its sign (without labels) so every computer draws the same map
    signs = np.sign(Vt[np.arange(k), np.argmax(np.abs(Vt[:k]), axis=1)])
    return U[:, :k] * S[:k] * signs, (S ** 2 / np.sum(S ** 2))[:k]


scores, share = pca_scores(X)                           # note: neither y nor the subtypes are used here
print(f"axis 1 keeps {share[0]:.1%} and axis 2 keeps {share[1]:.1%} of the variation in all 3,571 genes")
SUBTYPE_COLOURS = {"B-ALL": CLASS_COLORS[0], "T-ALL": CLASS_COLORS[2], "AML": CLASS_COLORS[1]}
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
a1.scatter(scores[:, 0], scores[:, 1], s=26, color="0.5", edgecolor="k", lw=0.3)
a1.set_title("what the algorithm sees: 72 SIMULATED patients, no labels")
for name, colour in SUBTYPE_COLOURS.items():
    m = leuk["subtype"] == name
    a2.scatter(scores[m, 0], scores[m, 1], s=26, color=colour, edgecolor="k", lw=0.3, label=name)
a2.set_title("the same map, coloured AFTERWARDS by the true subtype"); a2.legend()
for a in (a1, a2):
    a.set_xlabel("PC 1 (main axis of variation)"); a.set_ylabel("PC 2")
plt.tight_layout(); plt.show()
print("🧪 average coordinates by true subtype:")
for name in SUBTYPE_COLOURS:
    m = leuk["subtype"] == name
    print(f"   {name:6s} PC 1 = {scores[m, 0].mean():5.1f}   PC 2 = {scores[m, 1].mean():5.1f}")

🔍 **What just happened?** The left panel is all the algorithm has: 72 points with no labels — and you can already see
groups: a cloud on the left, a larger one on the right and a few points high up. Colouring by the true subtype (right
panel, 🧪 — possible only because the data are simulated) reveals what they are. The main axis of variation separates
**AML** (left, average PC-1 coordinate −5.6) from **ALL** (right: 3.2 for B-ALL, 2.1 for T-ALL). The second axis picks
out the **T-ALL** patients (average PC-2 coordinate 7.6 against −1.6 and −0.2) — a subtype nobody told the method
about. ALL indeed comes in two lineages, B-cell and T-cell, a distinction clinicians knew and the simulated data
contain. The two axes keep only about **8 %** and **5 %** of the total variation: most of the variation in 3,571 genes
is *not* about leukaemia type (patients differ in many other ways), yet the largest directions of variation carry the
disease signal.

⚠️ **Hypothesis-generating, not proof.** A low-dimensional picture can suggest groups, but it can also mislead: other
choices (scaling genes, removing a batch effect) give other pictures, and an apparent cluster may reflect the lab that
processed the samples rather than biology. Golub *et al.* used a clustering method (self-organising maps) for class
discovery and then *checked* the discovered classes against the known diagnosis. Clustering is Module 12; checking
whether discovered structure is real is a recurring theme of Modules 12–14 and 23.

### 5.2 Genes mirror geography — a simulated population

The most famous unsupervised picture in genetics is the PCA map of Europeans by Novembre *et al.* (2008): plotted
on the first two principal components of their genotypes, people from each country landed roughly where their country
lies on the map — "genes mirror geography". Why? Neighbours share ancestors, so allele frequencies change *gradually*
across a continent. We simulate exactly that: 600 people scattered over a square "continent", 2,000 SNPs whose allele
frequencies drift smoothly from west to east and from south to north, genotypes 0/1/2 drawn from those frequencies.
PCA receives only the genotypes — never the locations.

In [ ]:
# ── 5.2 Genes mirror geography: PCA of 600 SIMULATED people × 2,000 SNPs ──
pop_rng = np.random.default_rng(SEED)
n_people, n_snps = 600, 2000
east, north = pop_rng.uniform(0, 1, n_people), pop_rng.uniform(0, 1, n_people)    # where each person lives
centre_freq = pop_rng.uniform(0.1, 0.9, n_snps)                    # allele frequency in the middle of the map
drift_east = pop_rng.normal(0, 0.18, n_snps)                       # change in frequency from west to east ...
drift_north = pop_rng.normal(0, 0.12, n_snps)                      # ... and from south to north
freq = np.clip(centre_freq + np.outer(east - 0.5, drift_east) + np.outer(north - 0.5, drift_north), 0.01, 0.99)
G = pop_rng.binomial(2, freq)                                      # genotypes: copies of the allele (0, 1 or 2)
print("genotype table:", G.shape, "→", n_people, "people ×", n_snps, "SNPs; first person:", G[0, :12], "...")

pop_scores, pop_share = pca_scores(G)                              # the locations are NOT given to PCA
place_colour = np.c_[east, 0.45 * np.ones(n_people), north]        # colour = position (red = east, blue = north)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4.2))
a1.scatter(east, north, c=place_colour, s=12)
a1.set_xlabel("west → east"); a1.set_ylabel("south → north"); a1.set_title("the TRUE map of where people live")
a2.scatter(pop_scores[:, 0], pop_scores[:, 1], c=place_colour, s=12)
a2.set_xlabel(f"PC 1 ({pop_share[0]:.1%} of variation)"); a2.set_ylabel(f"PC 2 ({pop_share[1]:.1%})")
a2.set_title("SIMULATED genotypes → PCA (no locations used)")
plt.tight_layout(); plt.show()
print("🧪 correlation of PC 1 with east: {:+.2f}, of PC 2 with north: {:+.2f}".format(
    np.corrcoef(pop_scores[:, 0], east)[0, 1], np.corrcoef(pop_scores[:, 1], north)[0, 1]))

🔍 **What just happened?** The PCA map (right) looks like the true map (left), with the same colours in the same
arrangement — although PCA saw only 1.2 million genotypes and no coordinates. The 🧪 check puts numbers on it: PC 1
tracks east–west and PC 2 tracks south–north, each with a correlation close to ±1 (the sign of an axis is arbitrary,
so the map may come out mirrored — here north is at the bottom). The two axes keep only 2.7 % and 1.4 % of the
variation — most genotype variation is individual noise — yet those two directions contain the geography. In real data the picture is messier (migration,
uneven sampling, the shape of a real continent), but the principle is the same.

✏️ **Your turn.** In cell 5.2 make the north–south drift as strong as the east–west drift (`0.18` for both) and re-run.
The map now comes out *rotated*: PCA finds the two directions of largest variation, which no longer line up with the
compass. Why can PCA not know which way is north?

Finally, a classic diagram summarises the three kinds of learners met so far.

In [ ]:
# ── 5.3 Three kinds of learners: classifier and regressor (supervised), density estimator (unsupervised) ──
fig, ax = plt.subplots(figsize=(9, 3.9))
learners = [("Classifier", "prediction of a category\n(ALL or AML)", "supervised", "tab:orange"),
            ("Regressor", "prediction of a real number\n(a blood-pressure value)", "supervised", "tab:orange"),
            ("Density\nestimator", "a probability P(x)\n(how typical is this sample?)", "unsupervised", "tab:green")]
for r, (name, output, kind, colour) in enumerate(learners):
    yc = 2.4 - 1.2 * r
    ax.text(0.2, yc, "input\nattributes X", ha="center", va="center", fontsize=8, color="darkgreen")
    for dy in np.linspace(-0.28, 0.28, 5):                         # many input features ...
        ax.annotate("", xy=(2.0, yc + 0.5 * dy), xytext=(0.85, yc + dy),
                    arrowprops=dict(arrowstyle="-|>", lw=0.7, color="0.35"))
    ax.add_patch(FancyBboxPatch((2.05, yc - 0.3), 1.5, 0.6, boxstyle="round,pad=0.03", fc=colour, ec="0.3", alpha=0.45))
    ax.text(2.8, yc, name, ha="center", va="center", fontsize=8.5, fontweight="bold")
    ax.annotate("", xy=(4.3, yc), xytext=(3.62, yc), arrowprops=dict(arrowstyle="-|>", lw=1.2, color="0.3"))
    ax.text(4.4, yc, output, ha="left", va="center", fontsize=8, color="purple")
    ax.text(-0.75, yc, kind, ha="right", va="center", fontsize=8, style="italic", color=colour)
ax.axhline(0.6, xmin=0.02, xmax=0.98, color="0.6", lw=0.8, ls="--")
ax.set_xlim(-2.2, 6.6); ax.set_ylim(-0.55, 3.0); ax.axis("off")
ax.set_title("Supervised learners map X to Y; an unsupervised density estimator maps X to a probability")
plt.tight_layout(); plt.show()

🔍 **What just happened?** In supervised learning the data are input–output pairs; depending on whether the output is
discrete or continuous, the learner is a **classifier** or a **regressor**. In unsupervised learning there is only
input; the learner is a **density estimator** that says how probable (how typical) an input is — and patterns such as
groups or main directions are properties of that density.

### 📘 Lecture notes — unsupervised learning, and the two families side by side

**Definition.** Unsupervised learning performs the task "given X, find an interesting pattern" — there is no
label Y and no teacher. Examples: cluster images of animals or newspaper articles by topic; learn a gene
regulatory network from expression data alone (the unknown "output" is the network); find population structure
from genotypes (197,146 SNPs in 1,387 Europeans → 2 coordinates per person, no country labels used).

**Probabilistic formulation.** Unsupervised learning learns the density of the data,

$$P(X)$$

*In words:* a model of which inputs are common and which are rare. It is usually harder to learn than
$P(Y \mid X)$, because it must capture the differences among all samples; we make it learnable by assuming what
kind of pattern we expect (groups, a few main directions, a network) — that assumption is built into the model.

**Why do it?** Labels are expensive, unlabelled data cheap and plentiful. But you need many samples (a network
cannot be learned from one expression profile), and evaluation is hard: without correct answers, how do we know a
result is right?

| | Supervised | Unsupervised |
|---|---|---|
| data | pairs (X, Y) | X only |
| model | P(Y ∣ X) | P(X) |
| teacher | yes ("correct answer for Y") | no |
| learners | classifier (discrete Y), regressor (continuous Y) | density estimator |
| evaluation | compare predictions with true labels | hard — needs other evidence |

*Worked example (Section 5):* PCA of 72 leukaemia profiles, computed without labels, put AML and ALL at opposite
ends of axis 1 and T-ALL on axis 2; the simulated truth confirmed it.

> 🎓 **Exam tip.** The supervised/unsupervised distinction is about the **training data** (pairs (X, Y) vs X alone), not about
    the method's name. The same leukaemia data support a supervised task (class prediction) and an unsupervised one
    (class discovery).

---
## 6 · What is X? What is Y? What is f?

> 📍 **Where we are.** We now revisit the biological examples of Section 2 and ask, for each one, the three questions
> that start every machine-learning project. Cover the answers, decide for yourself, then click to reveal.

| Problem | Your guess: X? Y? f? Supervised or unsupervised? | Answer |
|---|---|---|
| Speech recognition | | <details><summary>reveal</summary>**X** = the sound recording; **Y** = the written sentence, one word per segment ("I love the Santa Cruz beach"); **f** = a speech recogniser (e.g. a hidden Markov model). Supervised: trained on recordings with transcripts.</details> |
| Gene finding | | <details><summary>reveal</summary>**X** = the DNA sequence (letters a, c, g, t); **Y** = a label for **every position** (exon, intron, intergenic); **f** = a hidden Markov model. Supervised when trained on annotated genes. Y is a whole *sequence* of labels. Modules 19–20.</details> |
| Genetic basis of disease / GWAS | | <details><summary>reveal</summary>**X** = a person's genotypes at many SNPs (0/1/2); **Y** = healthy vs diseased (classification) or a trait value (regression); **f** = logistic or linear regression. Supervised. Unlike gene finding, **one** label for the whole genome. Modules 06–11.</details> |
| Golub ALL vs AML | | <details><summary>reveal</summary>**X** = the expression of thousands of genes in one patient (a *column* of the classic heatmap, a *row* of our table); **Y** = ALL or AML; **f** = a classifier. Supervised (class prediction); finding the classes without labels is unsupervised (class discovery). Modules 08–14.</details> |
| Gene regulatory networks | | <details><summary>reveal</summary>**X** = expression of many genes in hundreds of individuals; **no Y is given** — the unknown "output" is the network: which genes depend on which. Unsupervised; a probabilistic graphical model learns P(X). Modules 15–18.</details> |
| Population structure in Europe | | <details><summary>reveal</summary>**X** = 197,146 genotypes per person for 1,387 people; **no Y** (country labels are *not* used); output = 2 coordinates per person (and the groups they form). Unsupervised dimensionality reduction (PCA). Module 14.</details> |

✏️ **Your turn.** Invent one problem from your own biology courses — for example *"predict a protein's subcellular
location from its amino-acid sequence"* or *"group single cells from a tumour into cell types"* — and write down X, Y
(if any), f, and whether it is supervised or unsupervised, regression or classification.

---
## 7 · The course map

> 📍 **Where we are.** You have seen the two families of problems. To finish, a map of *all* the methods of 02-620, the
> probabilistic approach that ties them together, and the order of the lectures.

**Methods × tasks.** Most methods can be placed in a table. "Generative" models describe how the data X (and Y) are
*produced*, P(X, Y) or P(X); "discriminative" models describe only P(Y | X); "non-probabilistic" methods output an
answer without a probability model. Probability models appear again and again — logistic regression, naïve Bayes,
graphical models, latent-variable models, and probabilistic views of neural networks. SVMs and trees are sometimes
criticised for giving no uncertainty about their predictions, yet they are powerful methods with other strengths.

| Task | Non-probabilistic | Probabilistic, generative | Probabilistic, discriminative |
|---|---|---|---|
| Regression (continuous Y) | least squares as pure curve fitting | — | linear regression, ridge, lasso (06–07) |
| Classification (discrete Y) | k-nearest neighbours (08), decision trees & random forests (10), support vector machines (11) | naïve Bayes (08) | logistic regression (09), neural networks (22) |
| Clustering | k-means, hierarchical clustering (12) | Gaussian mixture models + EM (13) | — |
| Dimensionality reduction | PCA (14) | (probabilistic versions of PCA) | — |
| Networks and sequences | — | Bayesian networks (15–17), Gaussian graphical models (18), hidden Markov models (19–20) | — |

Bayesian learning and MCMC (21) and model selection (23) apply to every row.

### 7.1 Supervised learning through a joint distribution

There is a second way to get $P(Y \mid X)$: learn the **joint distribution** of all variables, then condition on the
inputs. Suppose we want a function f: ⟨G, H⟩ → W that predicts whether a person is wealthy (W) from gender (G) and
weekly working hours (H) — a toy illustration, not a serious model. Equivalently we want $P(W \mid G, H)$. Recipe:
learn $P(W, G, H)$ from data (a table with one probability for each of the 2 × 2 × 2 combinations), then compute
$P(W \mid G, H)$ from it. Once you have the joint distribution, you can ask for the probability of *any* logical
combination of the variables. The 📘 box at the end of this section works through the numbers; Module 02 lets you
compute with the full table, and Module 05 develops the idea.

### 7.2 Two ways to estimate parameters: MLE and MAP (a preview)

Every probability model has unknown numbers — **parameters**, written $\theta$ ("theta") — that must be estimated from
data $D$. Two principles recur throughout the course:

* **Maximum likelihood estimation (MLE)**, the *frequentist* approach: choose the $\theta$ that makes the observed data
  most probable,

  $$\hat\theta_{\text{MLE}} = \arg\max_{\theta} P(D \mid \theta)$$

  *In words:* try every possible value of θ, compute how probable the data would be under it (the **likelihood**),
  and keep the value with the highest likelihood. It uses the data only — "objective".

* **Maximum a posteriori estimation (MAP)**, the *Bayesian* approach: combine the data with a **prior** belief
  $P(\theta)$ through **Bayes' rule**,

  $$P(\theta \mid D) = \frac{P(D \mid \theta)\, P(\theta)}{P(D)} \qquad\qquad \hat\theta_{\text{MAP}} = \arg\max_{\theta} P(D \mid \theta)\, P(\theta)$$

  *In words:* the **posterior** belief about θ after seeing the data equals the likelihood times the prior belief,
  divided by the probability of the data (which does not depend on θ, so it can be ignored when maximising). MAP
  keeps the θ with the highest posterior. Critics call the prior "subjective"; supporters note that it encodes real
  knowledge.

When the log-likelihood is a smooth function of θ, both maxima are found by taking the logarithm, differentiating with
respect to θ and setting the derivative to zero (Modules 04–05). Here we simply try a fine grid of values. The
question: *what fraction θ of acute-leukaemia patients at a clinic have AML?* The data: 11 AML among the 38 training
patients. Our prior belief before seeing the data: "around 20 %, rarely above 50 %".

In [ ]:
# ── 7.2 MLE vs MAP on a grid: what fraction θ of acute-leukaemia patients have AML? ──
theta = np.linspace(0.001, 0.999, 999)                  # candidate values of the parameter


def likelihood(theta: np.ndarray, k: int, n: int) -> np.ndarray:
    """P(D | θ) for k AML patients among n independent patients (in one particular order)."""
    return theta ** k * (1 - theta) ** (n - k)


prior = theta ** 4 * (1 - theta) ** 14                   # prior belief: peaks near 0.22, small above 0.5
k, n = int(y[train].sum()), int(train.sum())

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.4), sharey=True)
for ax, (kk, nn) in zip(axes, [(k, n), (10 * k, 10 * n)]):
    lik = likelihood(theta, kk, nn)
    post = lik * prior
    mle, map_ = theta[np.argmax(lik)], theta[np.argmax(post)]
    for curve, name, colour in [(prior, "prior P(θ)", "0.5"), (lik, "likelihood P(D | θ)", "tab:blue"),
                                (post, "posterior ∝ P(D | θ) P(θ)", "tab:red")]:
        ax.plot(theta, curve / curve.max(), color=colour, label=name)   # each curve scaled to a maximum of 1
    ax.axvline(mle, color="tab:blue", ls="--", lw=1); ax.axvline(map_, color="tab:red", ls=":", lw=1.5)
    ax.set_xlim(0, 0.7); ax.set_xlabel("θ = fraction of patients with AML")
    ax.set_title(f"{kk} AML among {nn} patients: MLE = {mle:.3f}, MAP = {map_:.3f}")
    print(f"data: {kk:3d} of {nn:3d} → MLE = {mle:.3f} (= {kk}/{nn}), MAP = {map_:.3f}")
axes[0].set_ylabel("height (each curve scaled to max 1)"); axes[0].legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** *Left:* with the 38 training patients, the likelihood peaks at **11/38 = 0.289** — the MLE is
simply the observed fraction. The prior pulls the estimate towards its own peak (about 0.22), so the posterior peaks
a little lower, at **MAP = 0.268**. *Right:* with ten times as much data (110 of 380) the likelihood is much narrower,
it dominates the prior, and MAP (0.286) nearly equals MLE (0.289). This is the general pattern: **with little data the
prior matters; with lots of data MAP approaches MLE.** Module 05 derives both estimates with calculus (the MAP here is
$(k + 4)/(n + 18)$) and shows when each is preferable — for example ridge regression (Module 07) is MAP in disguise.

**Three ideas that recur in every module — watch for them.**

1. **The two-step template.** *Step 1:* choose a probability model — P(Y | X; θ) for supervised problems, P(X; θ) for
   unsupervised ones — with unknown parameters θ. *Step 2:* estimate θ from data, by **maximum likelihood (MLE)**,
   **maximum a posteriori (MAP)** or a full Bayesian analysis (Section 7.2; Modules 05 and 21). Methods without a
   probability model (trees, k-means, SVMs) are flagged as such and paired with a probabilistic cousin.
2. **Conditional independence buys tractability.** A full probability table over many variables needs astronomically
   many numbers: predicting a disease from 30 SNPs with three genotypes each needs a table P(Y | X₁, …, X₃₀) with
   3³⁰ ≈ 2 × 10¹⁴ entries (Module 08). Assuming that some variables are independent once we know others shrinks this
   to a manageable number — the trick behind naïve Bayes, Bayesian networks, hidden Markov models and Gaussian
   graphical models (Module 02 explains "conditional independence").
3. **Hidden (latent) variables lead to the EM algorithm.** Often the most interesting quantity is never observed: the
   cluster a cell belongs to, whether a stretch of DNA is an exon. Mixture models, hidden Markov models and k-means all
   alternate between *guessing the hidden variables* and *re-estimating the parameters* (Modules 12, 13 and 20);
   Gibbs sampling (Module 21) is the sampling sibling of this idea.

### 7.3 The order of the 02-620 lectures

Finally, the week-by-week order in which 02-620 teaches the topics, with the modules that teach each week:

In [ ]:
# ── 7.3 The 02-620 schedule by topic, and the matching modules ──
schedule = pd.DataFrame([
    (1, "Introduction to ML", "supervised/unsupervised learning; MLE, MAP", "00, 05 (prerequisites 01–04)"),
    (2, "Regression", "linear regression, regularization", "06, 07"),
    (3, "Classification", "k-nearest neighbour; naïve Bayes; logistic regression", "08, 09"),
    (4, "Classification", "generative vs discriminative; decision tree, random forest", "09, 10"),
    (5, "Clustering", "hierarchical clustering, k-means", "12"),
    (6, "Classification", "SVM", "11"),
    (7, "Probabilistic graphical models", "Bayesian networks: models, learning, inference; Midterm I", "15, 16, 17"),
    (8, "—", "spring break", "—"),
    (9, "Latent variable models", "dynamic Bayesian networks, Gaussian graphical models", "18 (DBNs in 19–20)"),
    (10, "Latent variables; dimensionality reduction", "mixture models, EM; PCA", "13, 14"),
    (11, "Latent variable models", "hidden Markov model, forward–backward, Baum–Welch", "19, 20"),
    (12, "Latent variable models", "gene discovery from genome sequences; HMMs continued", "19, 20"),
    (13, "Bayesian learning; neural networks", "Bayesian estimation, MCMC; backpropagation", "21, 22"),
    (14, "Model selection", "bias–variance trade-off; Midterm II", "23"),
    (15, "Projects", "project presentations", "44"),
], columns=["week", "topic", "lecture topics", "BiologyToML modules"])
print(schedule.to_string(index=False))

🔍 **What just happened?** The table follows the order of the lectures: supervised methods in the first weeks, then
unsupervised ones from clustering to dimensionality reduction, then latent-variable models, Bayesian learning and
neural networks, and finally the general theory of model selection that applies to both families. Note that this
course reorders a few topics for learning's sake: SVMs (week 6) come right after the other classifiers as Module 11;
clustering (week 5) opens the unsupervised part as Module 12, followed by mixture models and PCA (week 10) as Modules
13–14, *before* the graphical models of weeks 7 and 9 (Modules 15–18). The 02-620 project — *analyse one data set with
three methods* — is rehearsed in the capstone, [Module 44](44_capstone_project.ipynb).

✏️ **Your turn.** Find in the table the week in which you will meet (a) the method behind the PCA maps of Section 5,
(b) the method Haussler borrowed from speech recognition, (c) the first method for predicting a continuous trait from
genotypes. Which module teaches each?

### 📘 Lecture notes — the probabilistic approach — joint distributions, MLE, MAP and Bayes' rule

**Supervised learning via the joint distribution.** To learn f: ⟨G, H⟩ → W, i.e. $P(W \mid G, H)$, learn the joint
$P(W, G, H)$ and condition:

$$P(W \mid G, H) = \frac{P(W, G, H)}{P(G, H)} = \frac{P(W, G, H)}{\sum_{w} P(W = w, G, H)}$$

*In words:* keep only the rows of the joint table that match the observed G and H, and rescale them so that they
sum to 1. *Worked example* with a classic census table of wealth (poor/rich), gender and hours worked
(≤ 40.5 or > 40.5 per week): P(female, > 40.5 h, poor) = 0.0421768 and P(female, > 40.5 h, rich) = 0.0116293, so
P(rich | female, > 40.5 h) = 0.0116293 / (0.0421768 + 0.0116293) = **0.216**.

**Parameters and two estimation principles.** A model family (e.g. Gaussian, Bernoulli) has unknown parameters θ;
"learning the distribution" means estimating θ from data D.

- **MLE** (frequentist, data only): $\hat\theta_{\text{MLE}} = \arg\max_\theta P(D \mid \theta)$ — *in words:*
  the θ under which the observed data are most probable. Recipe: write the log-likelihood, differentiate with
  respect to θ, set to 0.
- **Bayes' rule in machine learning:** $P(\theta \mid D) = P(D \mid \theta)\,P(\theta)\,/\,P(D)$ — *in words:*
  posterior (belief after seeing D) = likelihood (how probable D is under θ) × prior (belief before seeing D) ÷
  evidence. θ can be model parameters or predictions.
- **MAP** (Bayesian, data + belief): $\hat\theta_{\text{MAP}} = \arg\max_\theta P(D \mid \theta)\,P(\theta)$ —
  *in words:* the most probable θ after seeing the data; same recipe as MLE with log P(θ) added. Criticised as
  subjective, but useful when data are few.

*Worked example (Section 7.2):* 11 AML among 38 patients → MLE = 11/38 = 0.289; with a prior peaking near 0.22,
MAP = 15/56 = 0.268; with 110 of 380, MLE = 0.289 and MAP = 0.286 — more data, less influence of the prior.

**What you should know from the introductory lecture.** Supervised vs unsupervised learning; supervised learning
learns f: X → Y, often modelled as P(Y | X) to represent uncertainty; unsupervised learning learns the density P(X)
to extract patterns from X; MLE vs MAP.

> 🎓 **Exam tip.** Know which quantity each principle maximises: MLE maximises P(D | θ), MAP maximises P(D | θ) P(θ) — the
    evidence P(D) is constant in θ and drops out. A frequent mistake is to maximise P(θ | D) and P(D | θ) and call both
    "the likelihood".

## 📌 Key takeaways

- Machine learning grew where computer science meets statistics; its central idea is **learning from examples**, and
  the same methods serve vision, speech, text, robotics and biology.
- Biology became a data science through three breakthroughs — the **genome sequence** (where are the genes? → HMMs),
  **maps of human variation** (which variants cause disease? → regression/classification; ancestry → PCA) and
  **genome-wide expression profiling** (cancer subtypes → classification/clustering; gene networks → graphical models).
- Data are tables **X** with **N rows (samples)** and **J columns (features)**; biological tables are usually **wide**
  (J ≫ N), e.g. 72 patients × 3,571 genes.
- **Supervised learning** learns f: X → Y from labelled pairs; continuous Y = **regression**, discrete Y =
  **classification**; 02-620 models it as **P(Y | X)** to express uncertainty. Evaluate on **unseen** data.
- **Unsupervised learning** has only X and models **P(X)**: it finds structure (clusters, low-dimensional maps) that
  is hypothesis-generating and harder to evaluate.
- Parameters are estimated by **MLE** (maximise P(D | θ)) or **MAP** (maximise P(D | θ) P(θ), from **Bayes' rule**);
  with lots of data the two agree.
- Every module follows the same spine: **model → estimate (MLE/MAP)**, **conditional independence** to keep models
  small, **latent variables → EM**.
- Simulated data have a **known truth**: use it to check whether a method found what is really there.

---
## ✅ Checkpoint

1. A notebook cell fails with `NameError: name 'leuk' is not defined`. What most likely happened and how do you fix it?
   <details><summary>Answer</summary>The cell that creates `leuk` (cell 3.1) has not been run in this session — perhaps the kernel was restarted. Run the cells above in order (or *Restart and run all*).</details>
2. Is predicting a patient's blood pressure from 500,000 SNPs regression or classification? Supervised or unsupervised? What is N and what is J?
   <details><summary>Answer</summary>Regression (Y is continuous), supervised (we have measured blood pressures). N = number of individuals (rows), J = 500,000 SNPs (columns).</details>
3. The leukaemia table has 72 rows and 3,571 columns. How many numbers is that, and why is the shape more important than the total?
   <details><summary>Answer</summary>72 × 3,571 = 257,112 numbers. The shape matters because there are far more features than samples (J ≫ N): with only 72 patients, many genes will differ between groups by chance, and some methods (e.g. ordinary least squares) break down.</details>
4. A classifier returns P(AML | x) = 0.52 for one patient and 0.97 for another. Which prediction should a doctor double-check, and why?
   <details><summary>Answer</summary>The 0.52 patient: the model is almost maximally uncertain (0.5/0.5 is a coin toss). But a high probability is not a guarantee — in Section 4 an AML patient received P(AML|x) = 0.18 (82 % for ALL), a fairly confident wrong answer. A toy model's probability is not a calibrated clinical risk, so in practice both would be checked by other tests.</details>
5. Why did we compute the gene ranking and the z-score scaling on the 38 training patients only?
   <details><summary>Answer</summary>Because the 34 test patients must play the role of *new* patients. Using them to choose genes or scale data leaks information about the test set into the model and makes the test accuracy over-optimistic (selection bias, Module 23).</details>
6. In the PCA map of the patients, AML and ALL separate along axis 1 and T-ALL along axis 2. Does that prove that PCA 'knows' leukaemia biology?
   <details><summary>Answer</summary>No. PCA only finds the directions of largest variation; here the largest ones happen to be related to the subtypes (the simulator built them in, and the 🧪 check confirmed it). The map is hypothesis-generating; other sources of variation (batch, sample type) could dominate in other data.</details>
7. Gene finding and GWAS both take a DNA sequence as input. How do their outputs Y differ?
   <details><summary>Answer</summary>Gene finding outputs a label for **every position** (a sequence of labels: exon/intron/intergenic); GWAS-style disease prediction outputs **one** label (or number) for the whole genome of a person.</details>
8. At a SNP, 0.83 of the patients but 0.17 of the healthy people carry the less common base (Section 2.3). What is the MAF if the two groups are equally large, and why is six people per group not enough to call the SNP disease-associated?
   <details><summary>Answer</summary>Overall frequency (0.83 + 0.17)/2 = 0.50, so MAF = 0.5. With six people per group, chance alone produces large differences — positions 4 and 11 differed between the groups (0.33 vs 0.00 and 0.17) although the simulation gave both groups the same probabilities. Real association studies need many people and a statistical test that accounts for testing many SNPs (Module 06).</details>
9. With 11 AML among 38 patients, what is the MLE of the AML fraction θ? Why is the MAP estimate with a prior centred near 0.2 smaller, and what happens with ten times more data?
   <details><summary>Answer</summary>MLE = 11/38 = 0.289. MAP maximises likelihood × prior, so the prior pulls the estimate towards 0.2 (MAP = 0.268). With 110 of 380 the likelihood dominates and MAP (0.286) ≈ MLE.</details>

## 🧠 Questions & discussion

1. The gene-finding story suggests that a good way to make progress in biology is to recognise that your problem has been solved in another field. Which other fields' problems resemble (a) clustering single cells, (b) predicting drug response from a genome?
2. The leukaemia data have 72 patients and thousands of genes. What could go wrong if you tried every pair of genes and reported the pair with the best test accuracy?
3. A predicted probability is only useful if it is *calibrated*: among patients given 0.8, about 80 % should truly be AML. How could you check calibration, and how many patients would you need?
4. Unsupervised learning has no correct answers. Suggest two ways a biologist could still decide whether the clusters found in a single-cell experiment are real.
5. UK Biobank links genotypes and health records for 500,000 people. Name one supervised and one unsupervised question you could ask of it, and one ethical issue the data raise.
6. Our simulations had a known truth. What does a simulation *not* tell you about how a method will behave on real data?

## 🚀 Bonus — going further

- **Use more genes.** Replace the two genes of Section 4 by the 50 genes chosen in Section 3 (`X[:, chosen]`) and re-run the nearest-centroid classifier. Does test accuracy improve? (Keep using the training set only to choose and scale.)
- **Look at other axes.** Plot PC 2 against PC 3 (`pca_scores(X, k=3)`). Does any other structure appear? What might it be?
- **Change the simulation.** Call `simulate_leukemia_expression(seed=1)` instead of the default and re-run Sections 3–5. Which numbers change, and which conclusions stay the same?
- **Read the original paper.** Golub *et al.* (1999), Science 286:531–537 — compare their Fig. 3B with our heatmap and their prediction-strength idea with our probabilities.
- **Related notebooks in this repository.** Generative vs discriminative classifiers: [`../../GenerativeDiscriminativeClassification/`](../../GenerativeDiscriminativeClassification/); probability and uncertainty: [`../../FundamentalsLearningUncertainty/`](../../FundamentalsLearningUncertainty/); the sister course on cryo-electron tomography: [`../../../../cryoem/tutorial/BiologyToResearch/`](../../../../cryoem/tutorial/BiologyToResearch/).

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module 00 — Welcome — date:
Input: SIMULATED leukaemia data shaped like Golub et al. 1999 (72 patients × 3,571 genes, log10; 38 train / 34 test);
       a simulated gene, 12 simulated people at 5 SNPs, a simulated population of 600 people × 2,000 SNPs
Output: a 50-gene heatmap, a two-gene classifier with P(AML|x), PCA maps, MLE and MAP of the AML fraction
Method: signal-to-noise gene ranking; nearest-centroid classification; PCA (black box); grid search for MLE/MAP
Key equation: P(AML|x) = exp(−d²_AML/2) / [exp(−d²_ALL/2) + exp(−d²_AML/2)]
Key parameters: the two genes (GENE_A, GENE_B); the number of top genes (25 + 25); the prior of Section 7.2
Ground-truth check: how many top genes truly differ; do the PCA axes match subtypes / geography?
What changed when I modified the gene pair / the ranking scale / the drift strengths:
One failure mode: (e.g., choosing genes by test accuracy = selection bias)
Connection to the 02-620 topics (which idea of the lecture this illustrates):
```

## 📚 References

- Carnegie Mellon University 02-620 *Machine Learning for Scientists* (course design S. Kim; instructors M. Xu and M. Zhang) — this module teaches the material of its introductory lecture, *Machine learning in science*.
- Golub T. R., Slonim D. K., Tamayo P., Huard C., Gaasenbeek M., Mesirov J. P., Coller H., Loh M. L., Downing J. R., Caligiuri M. A., Bloomfield C. D., Lander E. S. (1999) Molecular classification of cancer: class discovery and class prediction by gene expression monitoring. *Science* 286:531–537.
- Dudoit S., Fridlyand J., Speed T. P. (2002) Comparison of discrimination methods for the classification of tumors using gene expression data. *Journal of the American Statistical Association* 97:77–87.
- Novembre J., Johnson T., Bryc K., Kutalik Z., Boyko A. R., Auton A., Indap A., King K. S., Bergmann S., Nelson M. R., Stephens M., Bustamante C. D. (2008) Genes mirror geography within Europe. *Nature* 456:98–101.
- International HapMap Consortium (2005) A haplotype map of the human genome. *Nature* 437:1299–1320.
- Pennisi E. (2011) Will computers crash genomics? *Science* 331:666–668.
- Murphy K. P. (2012) *Machine Learning: A Probabilistic Perspective*. MIT Press — Chapter 1 (introduction; supervised vs unsupervised learning).
- Hastie T., Tibshirani R., Friedman J. (2009) *The Elements of Statistical Learning*, 2nd ed. Springer — Chapter 1 (examples, including gene-expression data) and Chapter 2.
- Mitchell T. M. (1997) *Machine Learning*. McGraw-Hill — Chapter 1.

## ➡️ Next

[Module 01 · Python & NumPy for biologists](01_python_numpy_for_biologists.ipynb) — you have *run* code;
    now you will *write* it: DNA strings, the genetic code, genotype matrices, the leukaemia table in pandas, and your
    first plots.